# Coastal Erosion Analysis: Threshold Detection and Pattern Recognition

## Research Workflow Overview
1. **Annual Erosion Event Definition** - Using Net Shoreline Movement (NSM)
2. **Transect Aggregation** - Beach-scale erosion assessment
3. **Environmental Data Processing** - Feature engineering from oceanographic drivers
4. **Exploratory Data Analysis** - Driver characterization and regime identification
5. **Pattern Recognition** - Erosion driver classification
6. **Threshold Detection Models** - HMM, Random Forest, XGBoost
7. **Model Evaluation & Final Thresholds**

---
**Monsoon Year Definition**: April (Year N) to March (Year N+1)

In [1]:
import matplotlib
matplotlib.use('Agg')
# =============================================================================
# Section 1: Import Libraries and Configure Environment
# =============================================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture  # Alternative to HMM for state detection

# XGBoost and SHAP
import xgboost as xgb
import shap

# HMM - Using Gaussian Mixture as fallback if hmmlearn unavailable
try:
    from hmmlearn import hmm
    HMM_AVAILABLE = True
except ImportError:
    HMM_AVAILABLE = False
    print("⚠ hmmlearn not available. Using Gaussian Mixture Model as alternative for state detection.")

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

print("✓ Libraries loaded successfully")
print(f"✓ HMM Available: {HMM_AVAILABLE}")

✓ Libraries loaded successfully
✓ HMM Available: True


## Section 2: Load Shoreline Data (NSM) and Define Erosion Labels

In [2]:
# =============================================================================
# Section 2: Load Shoreline Data and Define Erosion Labels
# =============================================================================

# Define file paths
DATA_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads"
SHORELINE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\all_stat.csv"
WAVE_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Gobal_Ocain_waves_reanalysis.nc"
WIND_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Global Ocean Monthly Mean Sea Surface Wind and Stress from Scatterometer and Model.nc"
CURRENT_FILE = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\2000-2025_Current_Data(Physics_Reanalysis).nc"

# Load shoreline statistics (transect-based)
shoreline_df = pd.read_csv(SHORELINE_FILE)

print("="*60)
print("SHORELINE DATA SUMMARY (Transect-Based)")
print("="*60)
print(f"Shape: {shoreline_df.shape}")
print(f"\nColumns: {list(shoreline_df.columns)}")
print(f"\nFirst 10 transects:")
display(shoreline_df.head(10))

# =============================================================================
# Section 2.1: Convert Transect Data to YEARLY Shoreline Changes
# Using SCE_closest_year to assign each transect's measurement to a year
# =============================================================================

# Assign transects to years based on SCE_closest_year
shoreline_df['measurement_year'] = shoreline_df['SCE_closest_year']

# Calculate ANNUAL shoreline statistics by aggregating transects per year
shoreline_annual = shoreline_df.groupby('measurement_year').agg({
    'NSM': ['mean', 'median', 'std', 'min', 'max', 'count'],
    'EPR': ['mean', 'median', 'std'],
    'LRR': ['mean', 'median', 'std'],
    'SCE': ['mean', 'max'],
}).reset_index()

# Flatten column names
shoreline_annual.columns = ['_'.join(col).strip('_') for col in shoreline_annual.columns]
shoreline_annual = shoreline_annual.rename(columns={'measurement_year': 'year'})

# Rename for consistency
shoreline_annual = shoreline_annual.rename(columns={'NSM_mean': 'annual_NSM'})

# =============================================================================
# Create Erosion Labels: 
# Since ALL transects show erosion (negative NSM), we use RELATIVE thresholds:
# - Severe Erosion: NSM below 25th percentile (more negative = more erosion)
# - Moderate Erosion: NSM between 25th-75th percentile
# - Mild Erosion/Stable: NSM above 75th percentile (less negative)
# For binary classification: Severe+Moderate = 1 (Erosion), Mild = 0 (Stable)
# =============================================================================

# Calculate thresholds based on distribution
median_nsm = shoreline_annual['annual_NSM'].median()
q25_nsm = shoreline_annual['annual_NSM'].quantile(0.25)  # More erosion (more negative)
q75_nsm = shoreline_annual['annual_NSM'].quantile(0.75)  # Less erosion

print(f"\n📊 Annual NSM Distribution:")
print(f"   Min (most erosion): {shoreline_annual['annual_NSM'].min():.2f} m")
print(f"   Q25: {q25_nsm:.2f} m")
print(f"   Median: {median_nsm:.2f} m")
print(f"   Q75: {q75_nsm:.2f} m")
print(f"   Max (least erosion): {shoreline_annual['annual_NSM'].max():.2f} m")

# Define erosion thresholds - use median as boundary
EROSION_NSM_THRESHOLD = median_nsm  # Years below median = erosion years
EROSION_PCT_THRESHOLD = 50  # Percentage-based threshold

# Create erosion status using relative threshold
shoreline_annual['Erosion_Status'] = shoreline_annual['annual_NSM'].apply(
    lambda x: 'Severe_Erosion' if x < q25_nsm 
    else ('Moderate_Erosion' if x < q75_nsm else 'Mild_Erosion')
)

# Binary label: Below median = 1 (High Erosion Year), Above median = 0 (Low Erosion Year)
shoreline_annual['Erosion_Binary'] = np.where(
    shoreline_annual['annual_NSM'] < median_nsm, 1, 0
)

print(f"\n✅ Using RELATIVE threshold: median NSM = {median_nsm:.2f} m")
print(f"   Years with NSM < median → High Erosion (1)")
print(f"   Years with NSM >= median → Low Erosion (0)")

print("\n" + "="*60)
print("YEARLY SHORELINE CHANGES (Aggregated from Transects)")
print("="*60)
print(f"\nYears with data: {len(shoreline_annual)}")
print(f"Year range: {shoreline_annual['year'].min()} - {shoreline_annual['year'].max()}")
print(f"\nAnnual Erosion Status Distribution:")
print(shoreline_annual['Erosion_Status'].value_counts())
print(f"\nYearly Shoreline Data:")
display(shoreline_annual)

# Also keep original transect-level labels
shoreline_df['Erosion_Label'] = np.where(shoreline_df['NSM'] < 0, 'Erosion', 'Accretion')
shoreline_df['Erosion_Binary'] = np.where(shoreline_df['NSM'] < 0, 1, 0)

print("\n" + "="*60)
print("TRANSECT-LEVEL EROSION DISTRIBUTION")
print("="*60)
print(f"\nErosion Label Distribution:")
print(shoreline_df['Erosion_Label'].value_counts())
print(f"\nMean NSM: {shoreline_df['NSM'].mean():.2f} m")
print(f"Median NSM: {shoreline_df['NSM'].median():.2f} m")

SHORELINE DATA SUMMARY (Transect-Based)
Shape: (109, 20)

Columns: ['id', 'SCE', 'SCE_highest_unc', 'SCE_trend', 'SCE_closest_year', 'SCE_farthest_year', 'NSM', 'NSM_highest_unc', 'NSM_trend', 'EPR', 'EPR_unc', 'EPR_trend', 'LRR', 'LR2', 'LSE', 'LCI', 'WLR', 'WR2', 'WSE', 'WCI']

First 10 transects:


,id,SCE,SCE_highest_unc,SCE_trend,SCE_closest_year,SCE_farthest_year,NSM,NSM_highest_unc,NSM_trend,EPR,EPR_unc,EPR_trend,LRR,LR2,LSE,LCI,WLR,WR2,WSE,WCI
0,1,29.96,5,accreting,2025,2011,-7.25,5,eroding,-0.48,0.47,eroding,-0.41,0.06,8.30,1.59,-0.41,0.06,1.66,1.59
1,2,26.04,5,accreting,2013,2011,-5.35,5,eroding,-0.35,0.47,stable,-0.23,0.02,7.99,1.53,-0.23,0.02,1.60,1.53
2,3,27.76,5,accreting,2025,2011,-8.32,5,eroding,-0.55,0.47,eroding,-0.50,0.12,6.70,1.29,-0.50,0.12,1.34,1.29
3,4,20.03,5,accreting,2025,2011,-6.49,5,eroding,-0.43,0.47,stable,-0.45,0.22,4.27,0.82,-0.45,0.22,0.85,0.82
4,5,19.87,5,accreting,2025,2011,-6.34,5,eroding,-0.42,0.47,stable,-0.37,0.13,4.81,0.92,-0.37,0.13,0.96,0.92
5,6,17.87,5,accreting,2025,2011,-5.73,5,eroding,-0.38,0.47,stable,-0.40,0.20,3.92,0.75,-0.40,0.20,0.78,0.75
6,7,15.52,5,accreting,2025,2011,-5.09,5,eroding,-0.34,0.47,stable,-0.27,0.10,4.04,0.78,-0.27,0.10,0.81,0.78
7,8,15.60,5,accreting,2013,2016,-5.89,5,eroding,-0.39,0.47,stable,-0.12,0.01,5.64,1.08,-0.12,0.01,1.13,1.08
8,9,18.81,5,accreting,2013,2018,-8.35,5,eroding,-0.55,0.47,eroding,-0.07,0.00,6.46,1.24,-0.07,0.00,1.29,1.24
9,10,22.30,5,accreting,2013,2022,-7.88,5,eroding,-0.52,0.47,eroding,0.00,0.00,7.09,1.36,0.00,0.00,1.42,1.36



📊 Annual NSM Distribution:
   Min (most erosion): -12.23 m
   Q25: -6.91 m
   Median: -5.66 m
   Q75: -1.49 m
   Max (least erosion): 3.54 m

✅ Using RELATIVE threshold: median NSM = -5.66 m
   Years with NSM < median → High Erosion (1)
   Years with NSM >= median → Low Erosion (0)

YEARLY SHORELINE CHANGES (Aggregated from Transects)

Years with data: 16
Year range: 2010 - 2025

Annual Erosion Status Distribution:
Erosion_Status
Moderate_Erosion    8
Severe_Erosion      4
Mild_Erosion        4
Name: count, dtype: int64

Yearly Shoreline Data:


,year,annual_NSM,NSM_median,NSM_std,NSM_min,NSM_max,NSM_count,EPR_mean,EPR_median,EPR_std,LRR_mean,LRR_median,LRR_std,SCE_mean,SCE_max,Erosion_Status,Erosion_Binary
0,2010,-6.796667,-8.280,9.413067,-15.38,3.27,3,-0.796667,-0.820,0.235867,-0.003333,0.000,0.045092,29.413333,35.59,Moderate_Erosion,1
1,2011,-9.094444,-9.540,4.690409,-16.58,2.88,27,-0.718889,-0.650,0.154928,0.115556,0.120,0.099163,30.426667,34.49,Severe_Erosion,1
2,2012,2.850909,3.080,1.113530,1.10,4.28,11,-0.187273,-0.200,0.073361,0.090000,0.100,0.029665,19.713636,20.96,Mild_Erosion,0
3,2013,-2.571429,-5.350,5.474698,-8.35,3.94,7,-0.347143,-0.350,0.151186,-0.040000,-0.010,0.110151,22.001429,28.32,Moderate_Erosion,0
4,2014,-3.170000,-3.170,10.352043,-10.49,4.15,2,-0.490000,-0.490,0.296985,0.125000,0.125,0.148492,24.775000,33.35,Moderate_Erosion,0
5,2015,2.160000,2.090,0.759430,1.24,3.27,7,-0.117143,-0.140,0.102097,0.067143,0.060,0.028702,24.985714,27.46,Mild_Erosion,0
6,2016,-5.303333,-8.640,7.855764,-10.94,3.67,3,-0.643333,-0.630,0.080829,0.083333,0.030,0.119304,28.146667,31.94,Moderate_Erosion,0
7,2017,-5.526667,-9.010,8.042216,-11.24,3.67,3,-0.713333,-0.750,0.100167,0.086667,0.050,0.100167,28.773333,33.54,Moderate_Erosion,0
8,2018,-5.790000,-9.520,7.713929,-10.93,3.08,3,-0.756667,-0.720,0.148436,0.126667,0.100,0.122202,28.430000,32.58,Moderate_Erosion,1
9,2019,-7.263333,-12.110,8.638295,-12.39,2.71,3,-0.763333,-0.800,0.081445,0.120000,0.100,0.131149,28.230000,32.71,Severe_Erosion,1



TRANSECT-LEVEL EROSION DISTRIBUTION

Erosion Label Distribution:
Erosion_Label
Erosion      69
Accretion    40
Name: count, dtype: int64

Mean NSM: -5.09 m
Median NSM: -7.88 m


In [3]:
# =============================================================================
# Section 2.2: Transect Aggregation (Beach-Scale Assessment)
# =============================================================================

# Calculate beach-scale statistics
transect_stats = {
    'Total_Transects': len(shoreline_df),
    'Mean_NSM': shoreline_df['NSM'].mean(),
    'Median_NSM': shoreline_df['NSM'].median(),
    'Std_NSM': shoreline_df['NSM'].std(),
    'Pct_Eroding': (shoreline_df['Erosion_Binary'].sum() / len(shoreline_df)) * 100,
    'Max_Erosion': shoreline_df['NSM'].min(),  # Most negative = max erosion
    'Max_Accretion': shoreline_df['NSM'].max(),
    'Mean_EPR': shoreline_df['EPR'].mean(),
    'Mean_LRR': shoreline_df['LRR'].mean()
}

print("="*60)
print("BEACH-SCALE AGGREGATION RESULTS")
print("="*60)
for key, value in transect_stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

# Determine overall beach state
EROSION_THRESHOLD_PCT = 60  # % of transects that must be eroding
if transect_stats['Mean_NSM'] < 0 and transect_stats['Pct_Eroding'] > EROSION_THRESHOLD_PCT:
    beach_state = "EROSION"
else:
    beach_state = "STABLE/ACCRETION"
    
print(f"\n{'='*60}")
print(f"OVERALL BEACH STATE: {beach_state}")
print(f"{'='*60}")
print(f"Criteria: Mean NSM < 0 AND >60% transects eroding")
print(f"  - Mean NSM = {transect_stats['Mean_NSM']:.2f} m (< 0: {transect_stats['Mean_NSM'] < 0})")
print(f"  - % Eroding = {transect_stats['Pct_Eroding']:.1f}% (> 60%: {transect_stats['Pct_Eroding'] > 60})")

BEACH-SCALE AGGREGATION RESULTS
Total_Transects: 109
Mean_NSM: -5.09
Median_NSM: -7.88
Std_NSM: 6.27
Pct_Eroding: 63.30
Max_Erosion: -16.58
Max_Accretion: 4.28
Mean_EPR: -0.53
Mean_LRR: 0.03

OVERALL BEACH STATE: EROSION
Criteria: Mean NSM < 0 AND >60% transects eroding
  - Mean NSM = -5.09 m (< 0: True)
  - % Eroding = 63.3% (> 60%: True)


In [4]:
# =============================================================================
# Section 2.3: Export Yearly Shoreline Data for Frontend
# =============================================================================

# Prepare yearly shoreline data for frontend table
yearly_shoreline_export = []

for _, row in shoreline_annual.iterrows():
    yearly_shoreline_export.append({
        'year': int(row['year']),
        'annual_NSM': round(float(row['annual_NSM']), 3),
        'NSM_median': round(float(row['NSM_median']), 3),
        'NSM_std': round(float(row['NSM_std']), 3),
        'NSM_min': round(float(row['NSM_min']), 3),
        'NSM_max': round(float(row['NSM_max']), 3),
        'NSM_count': int(row['NSM_count']),
        'EPR_mean': round(float(row['EPR_mean']), 3),
        'EPR_median': round(float(row['EPR_median']), 3),
        'LRR_mean': round(float(row['LRR_mean']), 3),
        'LRR_median': round(float(row['LRR_median']), 3),
        'SCE_mean': round(float(row['SCE_mean']), 3),
        'SCE_max': round(float(row['SCE_max']), 3),
        'Erosion_Status': row['Erosion_Status'],
        'Erosion_Binary': int(row['Erosion_Binary'])
    })

print("="*60)
print("YEARLY SHORELINE DATA EXPORT")
print("="*60)
print(f"Exported {len(yearly_shoreline_export)} years of data")
print(f"\nSample (first 10 years):")
for item in yearly_shoreline_export[:10]:
    print(f"  Year {item['year']}: NSM={item['annual_NSM']:.3f}m, Status={item['Erosion_Status']}")

print(f"\n✓ Ready for frontend table display")

YEARLY SHORELINE DATA EXPORT
Exported 16 years of data

Sample (first 10 years):
  Year 2010: NSM=-6.797m, Status=Moderate_Erosion
  Year 2011: NSM=-9.094m, Status=Severe_Erosion
  Year 2012: NSM=2.851m, Status=Mild_Erosion
  Year 2013: NSM=-2.571m, Status=Moderate_Erosion
  Year 2014: NSM=-3.170m, Status=Moderate_Erosion
  Year 2015: NSM=2.160m, Status=Mild_Erosion
  Year 2016: NSM=-5.303m, Status=Moderate_Erosion
  Year 2017: NSM=-5.527m, Status=Moderate_Erosion
  Year 2018: NSM=-5.790m, Status=Moderate_Erosion
  Year 2019: NSM=-7.263m, Status=Severe_Erosion

✓ Ready for frontend table display


## Section 3: Load and Process Environmental Data (NetCDF)

In [5]:
# =============================================================================
# Section 3.1: Load Wave Data
# =============================================================================

# Load wave reanalysis data
wave_ds = xr.open_dataset(WAVE_FILE)
print("="*60)
print("WAVE DATA SUMMARY")
print("="*60)
print(f"Variables: {list(wave_ds.data_vars)}")
print(f"Dimensions: {dict(wave_ds.dims)}")
print(f"Time range: {wave_ds.time.values[0]} to {wave_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in wave_ds.data_vars:
    print(f"  {var}: {wave_ds[var].dims} - {wave_ds[var].attrs.get('long_name', 'N/A')}")

WAVE DATA SUMMARY
Variables: ['VHM0', 'VTM10', 'VTM02', 'VTPK', 'VMDR', 'VPED']
Dimensions: {'time': 73041, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T21:00:00.000000000 to 2025-03-31T21:00:00.000000000

Variable details:
  VHM0: ('time', 'latitude', 'longitude') - Spectral significant wave height (Hm0)
  VTM10: ('time', 'latitude', 'longitude') - Spectral moments (-1,0) wave period (Tm-10)
  VTM02: ('time', 'latitude', 'longitude') - Spectral moments (0,2) wave period (Tm02)
  VTPK: ('time', 'latitude', 'longitude') - Wave period at spectral peak / peak period (Tp)
  VMDR: ('time', 'latitude', 'longitude') - Mean wave direction from (Mdir)
  VPED: ('time', 'latitude', 'longitude') - Wave principal direction at spectral peak


In [6]:
# =============================================================================
# Section 3.2: Load Wind Data
# =============================================================================

# Load wind data
wind_ds = xr.open_dataset(WIND_FILE)
print("="*60)
print("WIND DATA SUMMARY")
print("="*60)
print(f"Variables: {list(wind_ds.data_vars)}")
print(f"Dimensions: {dict(wind_ds.dims)}")
print(f"Time range: {wind_ds.time.values[0]} to {wind_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in wind_ds.data_vars:
    print(f"  {var}: {wind_ds[var].dims} - {wind_ds[var].attrs.get('long_name', 'N/A')}")

WIND DATA SUMMARY
Variables: ['eastward_wind', 'northward_wind', 'wind_speed', 'northward_stress', 'wind_stress_magnitude', 'eastward_stress']
Dimensions: {'time': 300, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T00:00:00.000000000 to 2025-03-01T00:00:00.000000000

Variable details:
  eastward_wind: ('time', 'latitude', 'longitude') - Stress-equivalent wind eastward component at 10 m
  northward_wind: ('time', 'latitude', 'longitude') - Stress-equivalent wind northward component at 10 m
  wind_speed: ('time', 'latitude', 'longitude') - Stress-equivalent wind speed at 10 m
  northward_stress: ('time', 'latitude', 'longitude') - Surface wind stress northward component
  wind_stress_magnitude: ('time', 'latitude', 'longitude') - Surface wind stress magnitude
  eastward_stress: ('time', 'latitude', 'longitude') - Surface wind stress eastward component


In [7]:
# =============================================================================
# Section 3.3: Load Current Data
# =============================================================================

# Load current data
current_ds = xr.open_dataset(CURRENT_FILE)
print("="*60)
print("CURRENT DATA SUMMARY")
print("="*60)
print(f"Variables: {list(current_ds.data_vars)}")
print(f"Dimensions: {dict(current_ds.dims)}")
print(f"Time range: {current_ds.time.values[0]} to {current_ds.time.values[-1]}")
print(f"\nVariable details:")
for var in current_ds.data_vars:
    print(f"  {var}: {current_ds[var].dims} - {current_ds[var].attrs.get('long_name', 'N/A')}")

CURRENT DATA SUMMARY
Variables: ['uo', 'vo']
Dimensions: {'time': 9131, 'depth': 1, 'latitude': 1, 'longitude': 1}
Time range: 2000-04-01T00:00:00.000000000 to 2025-03-31T00:00:00.000000000

Variable details:
  uo: ('time', 'depth', 'latitude', 'longitude') - Eastward velocity
  vo: ('time', 'depth', 'latitude', 'longitude') - Northward velocity


## Section 3.4: Temporal Aggregation - Monsoon Year (April-March)

**Critical Step**: Aggregate environmental drivers to annual monsoon year resolution.

| Driver | Aggregation Strategy |
|--------|---------------------|
| Hm0 (Wave Height) | Annual maximum |
| Storm days (wave) | Count per year (Hm0 > threshold) |
| Cumulative wave energy | Annual cumulative sum |
| Wind speed | Annual max and mean |
| Wind stress | Annual mean |
| Current velocity | Annual max and cumulative |

In [8]:
# =============================================================================
# Section 3.4: Create MONTHLY Environmental Features (More Samples for Training)
# =============================================================================
# IMPORTANT: Instead of yearly aggregation (only 25 samples), we keep monthly data
# This gives ~300 samples for better model training
# Annual erosion labels will be assigned to each month within that year

def assign_monsoon_year(time):
    """
    Assign monsoon year: April (Year N) to March (Year N+1) → Monsoon Year N
    """
    month = pd.Timestamp(time).month
    year = pd.Timestamp(time).year
    if month >= 4:  # April onwards
        return year
    else:  # Jan-March belongs to previous year's monsoon
        return year - 1

# Process Wave Data - MONTHLY resolution
wave_df = wave_ds.to_dataframe().reset_index()
wave_df = wave_df.dropna(subset=['VHM0'])
wave_df['monsoon_year'] = wave_df['time'].apply(assign_monsoon_year)
wave_df['year_month'] = wave_df['time'].dt.to_period('M')

# Define storm threshold
STORM_WAVE_THRESHOLD = 2.0  # meters

# Calculate wave energy proxy
wave_df['wave_energy'] = wave_df['VHM0']**2 * wave_df['VTPK']
wave_df['is_storm_wave'] = (wave_df['VHM0'] > STORM_WAVE_THRESHOLD).astype(int)

# MONTHLY wave aggregation
wave_monthly = wave_df.groupby(['monsoon_year', 'year_month']).agg({
    'VHM0': ['max', 'mean', 'std'],
    'VTPK': ['max', 'mean'],
    'wave_energy': 'sum',
    'is_storm_wave': 'sum',
    'time': 'first'  # Keep a reference time
}).reset_index()

# Flatten column names
wave_monthly.columns = ['monsoon_year', 'year_month', 'Hm0_max', 'Hm0_mean', 'Hm0_std', 
                        'Tp_max', 'Tp_mean', 'CumWaveEnergy', 'StormDays_wave', 'time']

# Also create annual aggregation for reference
wave_annual = wave_df.groupby('monsoon_year').agg({
    'VHM0': ['max', 'mean', 'std'],
    'VTPK': ['max', 'mean'],
    'wave_energy': 'sum',
    'is_storm_wave': 'sum'
}).reset_index()
wave_annual.columns = ['monsoon_year', 'Hm0_max_annual', 'Hm0_mean_annual', 'Hm0_std_annual', 
                       'Tp_max_annual', 'Tp_mean_annual', 'CumWaveEnergy_annual', 'StormDays_wave_annual']

print("="*60)
print("WAVE DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(wave_monthly)}")
print(f"Annual samples: {len(wave_annual)}")
print(f"\nSample rate increase: {len(wave_monthly) / len(wave_annual):.1f}x more training data!")
display(wave_monthly.head(10))

WAVE DATA - MONTHLY AGGREGATION
Monthly samples: 300
Annual samples: 25

Sample rate increase: 12.0x more training data!


,monsoon_year,year_month,Hm0_max,Hm0_mean,Hm0_std,Tp_max,Tp_mean,CumWaveEnergy,StormDays_wave,time
0,2000,2000-04,1.61,1.157940,0.235656,19.010000,12.607468,3985.690918,0,2000-04-01 21:00:00
1,2000,2000-05,2.13,1.372298,0.291076,19.020000,11.192782,5574.808594,8,2000-05-01 00:00:00
2,2000,2000-06,2.37,1.968833,0.192841,18.870001,10.738958,10089.267578,99,2000-06-01 00:00:00
3,2000,2000-07,2.39,1.815807,0.316649,18.870001,11.776733,9369.439453,77,2000-07-01 00:00:00
4,2000,2000-08,2.62,1.809758,0.364251,20.010000,11.829799,9672.284180,70,2000-08-01 00:00:00
5,2000,2000-09,2.24,1.573333,0.225913,21.370001,13.026583,7739.982910,7,2000-09-01 00:00:00
6,2000,2000-10,2.04,1.291250,0.275156,20.889999,12.228629,5027.863281,1,2000-10-01 00:00:00
7,2000,2000-11,2.31,1.027208,0.240491,19.940001,14.296499,3934.641846,4,2000-11-01 00:00:00
8,2000,2000-12,2.39,0.953589,0.273063,19.430000,12.335121,2799.014648,5,2000-12-01 00:00:00
9,2000,2001-01,1.13,0.774919,0.138728,20.500000,13.347298,2101.081543,0,2001-01-01 00:00:00


In [9]:
# =============================================================================
# Section 3.5: Wind Data - MONTHLY Aggregation
# =============================================================================

# Process Wind Data - MONTHLY resolution
wind_df = wind_ds.to_dataframe().reset_index()
wind_df = wind_df.dropna(subset=['wind_speed'])
wind_df['monsoon_year'] = wind_df['time'].apply(assign_monsoon_year)
wind_df['year_month'] = wind_df['time'].dt.to_period('M')

# Define storm wind threshold
STORM_WIND_THRESHOLD = 10.0  # m/s
wind_df['is_storm_wind'] = (wind_df['wind_speed'] > STORM_WIND_THRESHOLD).astype(int)

# MONTHLY wind aggregation
wind_monthly = wind_df.groupby(['monsoon_year', 'year_month']).agg({
    'wind_speed': ['max', 'mean', 'std'],
    'wind_stress_magnitude': ['max', 'mean'],
    'eastward_wind': 'mean',
    'northward_wind': 'mean',
    'is_storm_wind': 'sum'
}).reset_index()

# Flatten column names
wind_monthly.columns = ['monsoon_year', 'year_month', 'WindMax', 'WindMean', 'WindStd',
                        'WindStressMax', 'WindStressMean', 
                        'WindEast_mean', 'WindNorth_mean', 'StormDays_wind']

# Annual aggregation for reference
wind_annual = wind_df.groupby('monsoon_year').agg({
    'wind_speed': ['max', 'mean', 'std'],
    'wind_stress_magnitude': ['max', 'mean'],
    'is_storm_wind': 'sum'
}).reset_index()
wind_annual.columns = ['monsoon_year', 'WindMax_annual', 'WindMean_annual', 'WindStd_annual',
                       'WindStressMax_annual', 'WindStressMean_annual', 'StormDays_wind_annual']

print("="*60)
print("WIND DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(wind_monthly)}")
display(wind_monthly.head(10))

WIND DATA - MONTHLY AGGREGATION
Monthly samples: 300


,monsoon_year,year_month,WindMax,WindMean,WindStd,WindStressMax,WindStressMean,WindEast_mean,WindNorth_mean,StormDays_wind
0,2000,2000-04,4.78,4.78,NaN,0.04,0.04,4.29,1.15,0
1,2000,2000-05,5.21,5.21,NaN,0.03,0.03,4.91,1.28,0
2,2000,2000-06,6.73,6.73,NaN,0.08,0.08,6.46,0.46,0
3,2000,2000-07,5.71,5.71,NaN,0.05,0.05,5.48,0.51,0
4,2000,2000-08,6.00,6.00,NaN,0.05,0.05,5.47,1.33,0
5,2000,2000-09,4.99,4.99,NaN,0.03,0.03,4.33,1.76,0
6,2000,2000-10,5.17,5.17,NaN,0.04,0.04,4.74,0.29,0
7,2000,2000-11,4.17,4.17,NaN,0.03,0.03,0.62,-1.57,0
8,2000,2000-12,5.64,5.64,NaN,0.07,0.07,0.74,-4.25,0
9,2000,2001-01,4.64,4.64,NaN,0.03,0.03,0.58,-3.51,0


In [10]:
# =============================================================================
# Section 3.6: Current Data - MONTHLY Aggregation
# =============================================================================

# Process Current Data - MONTHLY resolution
current_df = current_ds.to_dataframe().reset_index()
current_df = current_df.dropna(subset=['uo', 'vo'])
current_df['monsoon_year'] = current_df['time'].apply(assign_monsoon_year)
current_df['year_month'] = current_df['time'].dt.to_period('M')

# Calculate current magnitude
current_df['current_magnitude'] = np.sqrt(current_df['uo']**2 + current_df['vo']**2)

# MONTHLY current aggregation
current_monthly = current_df.groupby(['monsoon_year', 'year_month']).agg({
    'current_magnitude': ['max', 'mean', 'std', 'sum'],
    'uo': 'mean',
    'vo': 'mean'
}).reset_index()

# Flatten column names
current_monthly.columns = ['monsoon_year', 'year_month', 'UcurrMax', 'UcurrMean', 'UcurrStd', 
                           'CumCurrent', 'Ucurr_east_mean', 'Ucurr_north_mean']

# Annual aggregation for reference
current_annual = current_df.groupby('monsoon_year').agg({
    'current_magnitude': ['max', 'mean', 'std', 'sum']
}).reset_index()
current_annual.columns = ['monsoon_year', 'UcurrMax_annual', 'UcurrMean_annual', 
                          'UcurrStd_annual', 'CumCurrent_annual']

print("="*60)
print("CURRENT DATA - MONTHLY AGGREGATION")
print("="*60)
print(f"Monthly samples: {len(current_monthly)}")
display(current_monthly.head(10))

CURRENT DATA - MONTHLY AGGREGATION
Monthly samples: 300


,monsoon_year,year_month,UcurrMax,UcurrMean,UcurrStd,CumCurrent,Ucurr_east_mean,Ucurr_north_mean
0,2000,2000-04,0.321304,0.174807,0.064589,5.244216,0.100731,-0.112003
1,2000,2000-05,0.258558,0.130523,0.059600,4.046205,0.072713,-0.092481
2,2000,2000-06,0.401125,0.208043,0.077728,6.241299,0.125899,-0.162521
3,2000,2000-07,0.419346,0.177735,0.108707,5.509781,0.097482,-0.128197
4,2000,2000-08,0.359857,0.183751,0.089079,5.696272,0.068854,-0.054874
5,2000,2000-09,0.400858,0.222400,0.090340,6.672010,0.153712,-0.136642
6,2000,2000-10,0.288950,0.125059,0.070878,3.876829,0.057414,-0.049263
7,2000,2000-11,0.415590,0.216157,0.092611,6.484716,0.012920,-0.074933
8,2000,2000-12,0.261459,0.147187,0.066616,4.562797,-0.055445,-0.046546
9,2000,2001-01,0.224856,0.112684,0.040024,3.493202,-0.073441,-0.019512


In [11]:
# =============================================================================
# Section 3.7: Merge Monthly Environmental Features + Annual Context
# =============================================================================

# Merge all MONTHLY datasets
env_features_monthly = wave_monthly.merge(wind_monthly, on=['monsoon_year', 'year_month'], how='outer')
env_features_monthly = env_features_monthly.merge(current_monthly, on=['monsoon_year', 'year_month'], how='outer')

# Merge annual features (provides annual context to monthly data)
env_features_monthly = env_features_monthly.merge(wave_annual, on='monsoon_year', how='left')
env_features_monthly = env_features_monthly.merge(wind_annual, on='monsoon_year', how='left')
env_features_monthly = env_features_monthly.merge(current_annual, on='monsoon_year', how='left')

# Filter to complete years (2000-2024)
env_features_monthly = env_features_monthly[
    (env_features_monthly['monsoon_year'] >= 2000) & 
    (env_features_monthly['monsoon_year'] <= 2024)
]

# Add month indicator for seasonality
env_features_monthly['month'] = env_features_monthly['year_month'].dt.month

# Create seasonal indicators (monsoon seasonality)
def get_season(month):
    if month in [6, 7, 8, 9]:  # SW Monsoon
        return 'SW_Monsoon'
    elif month in [10, 11]:  # NE Monsoon onset
        return 'NE_Monsoon'
    elif month in [12, 1, 2]:  # NE Monsoon peak
        return 'NE_Peak'
    else:  # Pre-monsoon
        return 'Pre_Monsoon'

env_features_monthly['season'] = env_features_monthly['month'].apply(get_season)

# Create dummy variables for seasons
season_dummies = pd.get_dummies(env_features_monthly['season'], prefix='Season')
env_features_monthly = pd.concat([env_features_monthly, season_dummies], axis=1)

print("="*60)
print("MONTHLY ENVIRONMENTAL FEATURE MATRIX")
print("="*60)
print(f"Shape: {env_features_monthly.shape}")
print(f"\n✓ {len(env_features_monthly)} monthly samples (vs ~25 annual samples)")
print(f"✓ ~{len(env_features_monthly)/25:.0f}x more training data!")
print(f"\nMonsoon years covered: {env_features_monthly['monsoon_year'].min()} to {env_features_monthly['monsoon_year'].max()}")
print(f"Total months: {len(env_features_monthly)}")

# Check for missing values
missing = env_features_monthly.isnull().sum()
if missing.sum() > 0:
    print(f"\n⚠ Missing values detected:")
    print(missing[missing > 0])

print(f"\nFeatures ({len(env_features_monthly.columns)} total):")
for col in env_features_monthly.columns[:15]:
    print(f"  - {col}")
print(f"  ... and {len(env_features_monthly.columns)-15} more")

display(env_features_monthly.head(10))

MONTHLY ENVIRONMENTAL FEATURE MATRIX
Shape: (300, 47)

✓ 300 monthly samples (vs ~25 annual samples)
✓ ~12x more training data!

Monsoon years covered: 2000 to 2024
Total months: 300

⚠ Missing values detected:
WindStd                  300
WindStressMax            219
WindStressMean           219
WindStressMax_annual     216
WindStressMean_annual    216
dtype: int64

Features (47 total):
  - monsoon_year
  - year_month
  - Hm0_max
  - Hm0_mean
  - Hm0_std
  - Tp_max
  - Tp_mean
  - CumWaveEnergy
  - StormDays_wave
  - time
  - WindMax
  - WindMean
  - WindStd
  - WindStressMax
  - WindStressMean
  ... and 32 more


,monsoon_year,year_month,Hm0_max,Hm0_mean,Hm0_std,Tp_max,Tp_mean,CumWaveEnergy,StormDays_wave,time,...,UcurrMax_annual,UcurrMean_annual,UcurrStd_annual,CumCurrent_annual,month,season,Season_NE_Monsoon,Season_NE_Peak,Season_Pre_Monsoon,Season_SW_Monsoon
0,2000,2000-04,1.61,1.157940,0.235656,19.010000,12.607468,3985.690918,0,2000-04-01 21:00:00,...,0.419346,0.160645,0.084027,58.635483,4,Pre_Monsoon,False,False,True,False
1,2000,2000-05,2.13,1.372298,0.291076,19.020000,11.192782,5574.808594,8,2000-05-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,5,Pre_Monsoon,False,False,True,False
2,2000,2000-06,2.37,1.968833,0.192841,18.870001,10.738958,10089.267578,99,2000-06-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,6,SW_Monsoon,False,False,False,True
3,2000,2000-07,2.39,1.815807,0.316649,18.870001,11.776733,9369.439453,77,2000-07-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,7,SW_Monsoon,False,False,False,True
4,2000,2000-08,2.62,1.809758,0.364251,20.010000,11.829799,9672.284180,70,2000-08-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,8,SW_Monsoon,False,False,False,True
5,2000,2000-09,2.24,1.573333,0.225913,21.370001,13.026583,7739.982910,7,2000-09-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,9,SW_Monsoon,False,False,False,True
6,2000,2000-10,2.04,1.291250,0.275156,20.889999,12.228629,5027.863281,1,2000-10-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,10,NE_Monsoon,True,False,False,False
7,2000,2000-11,2.31,1.027208,0.240491,19.940001,14.296499,3934.641846,4,2000-11-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,11,NE_Monsoon,True,False,False,False
8,2000,2000-12,2.39,0.953589,0.273063,19.430000,12.335121,2799.014648,5,2000-12-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,12,NE_Peak,False,True,False,False
9,2000,2001-01,1.13,0.774919,0.138728,20.500000,13.347298,2101.081543,0,2001-01-01 00:00:00,...,0.419346,0.160645,0.084027,58.635483,1,NE_Peak,False,True,False,False


In [12]:
# =============================================================================
# Section 3.8: Handle Missing Values and Merge with ACTUAL Shoreline Data
# =============================================================================

# Fill missing values using interpolation
numeric_cols = env_features_monthly.select_dtypes(include=[np.number]).columns
env_features_monthly[numeric_cols] = env_features_monthly[numeric_cols].interpolate(method='linear')
env_features_monthly = env_features_monthly.ffill().bfill()

# =============================================================================
# IMPORTANT: Use ACTUAL YEARLY SHORELINE DATA (from SCE_closest_year)
# Environmental data = MONTHLY (env_features_monthly)
# Shoreline data = YEARLY (shoreline_annual)
# =============================================================================

# Calculate normalized intensity scores (for monthly data)
env_features_monthly['wave_intensity'] = (
    (env_features_monthly['Hm0_max'] - env_features_monthly['Hm0_max'].mean()) / 
    env_features_monthly['Hm0_max'].std()
)
env_features_monthly['wind_intensity'] = (
    (env_features_monthly['WindMax'] - env_features_monthly['WindMax'].mean()) / 
    env_features_monthly['WindMax'].std()
)
env_features_monthly['current_intensity'] = (
    (env_features_monthly['UcurrMax'] - env_features_monthly['UcurrMax'].mean()) / 
    env_features_monthly['UcurrMax'].std()
)

# Monthly environmental forcing index
env_features_monthly['env_forcing_index'] = (
    env_features_monthly['wave_intensity'] + 
    env_features_monthly['wind_intensity'] + 
    env_features_monthly['current_intensity']
) / 3

# =============================================================================
# Merge ACTUAL yearly shoreline erosion labels with monthly environmental data
# Each year's REAL erosion status is assigned to all 12 months in that year
# =============================================================================

# Prepare shoreline_annual for merging (rename year to monsoon_year)
shoreline_annual_merge = shoreline_annual.copy()
shoreline_annual_merge = shoreline_annual_merge.rename(columns={'year': 'monsoon_year'})

# Merge actual shoreline data with monthly environmental data
env_features_monthly = env_features_monthly.merge(
    shoreline_annual_merge[['monsoon_year', 'annual_NSM', 'Erosion_Binary', 'Erosion_Status', 'NSM_count']], 
    on='monsoon_year', 
    how='left',
    suffixes=('_env', '_shore')
)

# Use actual shoreline erosion labels
if 'Erosion_Binary_shore' in env_features_monthly.columns:
    env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Binary_shore']
    env_features_monthly = env_features_monthly.drop(columns=['Erosion_Binary_shore'], errors='ignore')
elif 'Erosion_Binary' in env_features_monthly.columns:
    env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Binary']

# Handle years without shoreline data (fill with 0 - stable)
env_features_monthly['Erosion_Label'] = env_features_monthly['Erosion_Label'].fillna(0).astype(int)
env_features_monthly['annual_NSM'] = env_features_monthly['annual_NSM'].fillna(0)

print("="*60)
print("DATA FREQUENCY SUMMARY")
print("="*60)
print(f"\n📊 ENVIRONMENTAL DATA: MONTHLY Frequency")
print(f"   - {len(env_features_monthly)} monthly samples")
print(f"   - Features: Wave (Hm0), Wind, Current per month")

print(f"\n📊 SHORELINE DATA: YEARLY Frequency")
print(f"   - {len(shoreline_annual)} yearly samples")
print(f"   - Features: NSM, EPR, LRR aggregated from transects")

print(f"\n🔗 MERGED DATA: Monthly env + Yearly erosion labels")
print(f"   - Each month gets its year's erosion status")
print(f"   - Years with shoreline data: {env_features_monthly['monsoon_year'].nunique()}")

print(f"\n✓ ACTUAL Shoreline Erosion Labels (from SCE_closest_year):")
print(f"   Erosion months: {env_features_monthly['Erosion_Label'].sum()} ({env_features_monthly['Erosion_Label'].mean()*100:.1f}%)")
print(f"   Stable/Accretion months: {len(env_features_monthly) - env_features_monthly['Erosion_Label'].sum()}")

# Show which years have actual shoreline data
years_with_shore_data = env_features_monthly[env_features_monthly['annual_NSM'] != 0]['monsoon_year'].unique()
print(f"\n📅 Years with ACTUAL shoreline measurements: {sorted(years_with_shore_data)}")

# =============================================================================
# Create annual aggregated environmental data (env_features)
# =============================================================================

# Calculate annual statistics for each year
annual_stats = env_features_monthly.groupby('monsoon_year').agg({
    'Hm0_max': 'max',   # Annual max wave height
    'WindMax': 'max',   # Annual max wind
    'UcurrMax': 'max',  # Annual max current
    'env_forcing_index': 'max',  # Max monthly forcing that year
    'Erosion_Label': 'first',  # Use actual yearly erosion label
    'annual_NSM': 'first'  # Actual NSM for the year
}).reset_index()

annual_stats.columns = ['monsoon_year', 'Hm0_max_year', 'WindMax_year', 
                        'UcurrMax_year', 'max_forcing', 'Erosion_Label', 'annual_NSM']

# Create annual normalized scores
annual_stats['annual_wave_norm'] = (
    (annual_stats['Hm0_max_year'] - annual_stats['Hm0_max_year'].mean()) / 
    annual_stats['Hm0_max_year'].std()
)
annual_stats['annual_wind_norm'] = (
    (annual_stats['WindMax_year'] - annual_stats['WindMax_year'].mean()) / 
    annual_stats['WindMax_year'].std()
)
annual_stats['annual_curr_norm'] = (
    (annual_stats['UcurrMax_year'] - annual_stats['UcurrMax_year'].mean()) / 
    annual_stats['UcurrMax_year'].std()
)

annual_stats['annual_forcing_composite'] = (
    annual_stats['annual_wave_norm'] + 
    annual_stats['annual_wind_norm'] + 
    annual_stats['annual_curr_norm']
) / 3

# Create env_features (annual) for backward compatibility
env_features = annual_stats.copy()

# Add more annual features from the original annual aggregations
env_features = env_features.merge(wave_annual, on='monsoon_year', how='left')
env_features = env_features.merge(wind_annual, on='monsoon_year', how='left')
env_features = env_features.merge(current_annual, on='monsoon_year', how='left')

# Create backward compatible column names
env_features['Hm0_max'] = env_features['Hm0_max_year']
env_features['WindMax'] = env_features['WindMax_year']
env_features['UcurrMax'] = env_features['UcurrMax_year']

# Fill available columns
for col in ['Hm0_mean', 'WindMean', 'UcurrMean', 'CumCurrent', 'WindStressMean', 
            'StormDays_wave', 'CumWaveEnergy']:
    if col + '_annual' in env_features.columns:
        env_features[col] = env_features[col + '_annual']
    elif col not in env_features.columns:
        env_features[col] = 0

# Fill NaN
env_features = env_features.fillna(0)

# Add intensity scores
env_features['wave_intensity'] = env_features['annual_wave_norm']
env_features['wind_intensity'] = env_features['annual_wind_norm'] 
env_features['current_intensity'] = env_features['annual_curr_norm']
env_features['env_forcing_index'] = env_features['annual_forcing_composite']

print(f"\n" + "="*60)
print("FINAL DATA STRUCTURE SUMMARY")
print("="*60)
print(f"\n📊 env_features_monthly (MONTHLY Environmental + Yearly Erosion Labels):")
print(f"   - Rows: {len(env_features_monthly)} monthly samples")
print(f"   - Each row: environmental data for one month")
print(f"   - Erosion_Label: ACTUAL yearly erosion from shoreline_annual")
print(f"   - Columns: {list(env_features_monthly.columns)[:10]}...")

print(f"\n📊 env_features (ANNUAL Environmental + Yearly Erosion Labels):")
print(f"   - Rows: {len(env_features)} yearly samples")
print(f"   - Columns: {list(env_features.columns)[:10]}...")

print(f"\n📊 shoreline_annual (YEARLY Shoreline from SCE_closest_year):")
print(f"   - Rows: {len(shoreline_annual)} years")
print(f"   - Features: annual_NSM, EPR_mean, LRR_mean, Erosion_Binary")

display(env_features_monthly[['monsoon_year', 'year_month', 'Hm0_max', 'WindMax', 
                              'UcurrMax', 'annual_NSM', 'Erosion_Label']].head(15))

DATA FREQUENCY SUMMARY

📊 ENVIRONMENTAL DATA: MONTHLY Frequency
   - 300 monthly samples
   - Features: Wave (Hm0), Wind, Current per month

📊 SHORELINE DATA: YEARLY Frequency
   - 16 yearly samples
   - Features: NSM, EPR, LRR aggregated from transects

🔗 MERGED DATA: Monthly env + Yearly erosion labels
   - Each month gets its year's erosion status
   - Years with shoreline data: 25

✓ ACTUAL Shoreline Erosion Labels (from SCE_closest_year):
   Erosion months: 84 (28.0%)
   Stable/Accretion months: 216

📅 Years with ACTUAL shoreline measurements: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

FINAL DATA STRUCTURE SUMMARY

📊 env_features_monthly (MONTHLY Environmental + Yearly Erosion Labels):
   - Rows: 300 monthly samples
   - Each row: environmental data for one month
   - Erosion_Label: AC

,monsoon_year,year_month,Hm0_max,WindMax,UcurrMax,annual_NSM,Erosion_Label
0,2000,2000-04,1.61,4.78,0.321304,0.0,0
1,2000,2000-05,2.13,5.21,0.258558,0.0,0
2,2000,2000-06,2.37,6.73,0.401125,0.0,0
3,2000,2000-07,2.39,5.71,0.419346,0.0,0
4,2000,2000-08,2.62,6.00,0.359857,0.0,0
5,2000,2000-09,2.24,4.99,0.400858,0.0,0
6,2000,2000-10,2.04,5.17,0.288950,0.0,0
7,2000,2000-11,2.31,4.17,0.415590,0.0,0
8,2000,2000-12,2.39,5.64,0.261459,0.0,0
9,2000,2001-01,1.13,4.64,0.224856,0.0,0


## Section 4: Exploratory Data Analysis (EDA)

Analyze the relationship between environmental drivers and erosion events:
- Boxplots comparing erosion vs stable years
- Correlation matrices
- PCA for regime identification
- Clustering analysis

In [13]:
# =============================================================================
# Section 4.1: Boxplots - Erosion vs Stable Years
# =============================================================================

# Key drivers to compare (using columns available in annual env_features)
# Check available columns and use only existing ones
available_cols = env_features.columns.tolist()
print("Available columns in env_features:", available_cols)

# Define key drivers that exist in the data
key_drivers = []
potential_drivers = ['Hm0_max', 'CumWaveEnergy', 'StormDays_wave', 
                     'WindMax', 'WindStressMean', 'UcurrMax', 'CumCurrent', 
                     'Hm0_mean', 'WindMean', 'UcurrMean']

for driver in potential_drivers:
    if driver in available_cols:
        key_drivers.append(driver)
    elif driver + '_annual' in available_cols:
        key_drivers.append(driver + '_annual')

print(f"\nUsing drivers: {key_drivers}")

# Limit to 8 drivers for the subplot layout
key_drivers = key_drivers[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 10))
axes = axes.flatten()

for i, driver in enumerate(key_drivers):
    ax = axes[i]
    erosion_data = env_features[env_features['Erosion_Label'] == 1][driver]
    stable_data = env_features[env_features['Erosion_Label'] == 0][driver]
    
    bp = ax.boxplot([stable_data, erosion_data], 
                    labels=['Stable', 'Erosion'],
                    patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgreen')
    bp['boxes'][1].set_facecolor('salmon')
    
    ax.set_title(driver.replace('_annual', ''))
    ax.set_ylabel('Value')

# Hide unused axes if less than 8 drivers
for j in range(len(key_drivers), 8):
    axes[j].set_visible(False)
    
plt.suptitle('Environmental Drivers: Erosion vs Stable Years', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical summary
print("="*60)
print("STATISTICAL COMPARISON: Erosion vs Stable Years")
print("="*60)
for driver in key_drivers:
    erosion_mean = env_features[env_features['Erosion_Label'] == 1][driver].mean()
    stable_mean = env_features[env_features['Erosion_Label'] == 0][driver].mean()
    diff_pct = ((erosion_mean - stable_mean) / stable_mean) * 100 if stable_mean != 0 else 0
    print(f"{driver.replace('_annual', ''):25s}: Erosion={erosion_mean:.3f}, Stable={stable_mean:.3f}, Diff={diff_pct:+.1f}%")

Available columns in env_features: ['monsoon_year', 'Hm0_max_year', 'WindMax_year', 'UcurrMax_year', 'max_forcing', 'Erosion_Label', 'annual_NSM', 'annual_wave_norm', 'annual_wind_norm', 'annual_curr_norm', 'annual_forcing_composite', 'Hm0_max_annual', 'Hm0_mean_annual', 'Hm0_std_annual', 'Tp_max_annual', 'Tp_mean_annual', 'CumWaveEnergy_annual', 'StormDays_wave_annual', 'WindMax_annual', 'WindMean_annual', 'WindStd_annual', 'WindStressMax_annual', 'WindStressMean_annual', 'StormDays_wind_annual', 'UcurrMax_annual', 'UcurrMean_annual', 'UcurrStd_annual', 'CumCurrent_annual', 'Hm0_max', 'WindMax', 'UcurrMax', 'Hm0_mean', 'WindMean', 'UcurrMean', 'CumCurrent', 'WindStressMean', 'StormDays_wave', 'CumWaveEnergy', 'wave_intensity', 'wind_intensity', 'current_intensity', 'env_forcing_index']

Using drivers: ['Hm0_max', 'CumWaveEnergy', 'StormDays_wave', 'WindMax', 'WindStressMean', 'UcurrMax', 'CumCurrent', 'Hm0_mean', 'WindMean', 'UcurrMean']


STATISTICAL COMPARISON: Erosion vs Stable Years
Hm0_max                  : Erosion=3.024, Stable=2.748, Diff=+10.1%
CumWaveEnergy            : Erosion=59714.094, Stable=57980.172, Diff=+3.0%
StormDays_wave           : Erosion=185.286, Stable=153.389, Diff=+20.8%
WindMax                  : Erosion=6.596, Stable=6.477, Diff=+1.8%
WindStressMean           : Erosion=0.000, Stable=0.015, Diff=-100.0%
UcurrMax                 : Erosion=0.484, Stable=0.463, Diff=+4.4%
CumCurrent               : Erosion=53.995, Stable=54.133, Diff=-0.3%
Hm0_mean                 : Erosion=1.199, Stable=1.186, Diff=+1.1%


In [14]:
# =============================================================================
# Section 4.1.1: Export Boxplot Data for Frontend
# =============================================================================

# Prepare boxplot comparison data for frontend
boxplot_export = []

for driver in key_drivers:
    erosion_data = env_features[env_features['Erosion_Label'] == 1][driver]
    stable_data = env_features[env_features['Erosion_Label'] == 0][driver]
    
    # Calculate statistics
    erosion_mean = erosion_data.mean()
    stable_mean = stable_data.mean()
    diff_pct = ((erosion_mean - stable_mean) / stable_mean) * 100 if stable_mean != 0 else 0
    
    # Calculate boxplot statistics for both groups
    stable_stats = {
        'min': float(stable_data.min()),
        'q1': float(stable_data.quantile(0.25)),
        'median': float(stable_data.median()),
        'q3': float(stable_data.quantile(0.75)),
        'max': float(stable_data.max()),
        'mean': float(stable_mean)
    }
    
    erosion_stats = {
        'min': float(erosion_data.min()),
        'q1': float(erosion_data.quantile(0.25)),
        'median': float(erosion_data.median()),
        'q3': float(erosion_data.quantile(0.75)),
        'max': float(erosion_data.max()),
        'mean': float(erosion_mean)
    }
    
    boxplot_export.append({
        'name': driver.replace('_annual', ''),
        'stableMean': float(stable_mean),
        'erosionMean': float(erosion_mean),
        'diffPct': float(diff_pct),
        'comparison': [
            {
                'category': 'Stable',
                'min': stable_stats['min'],
                'q1': stable_stats['q1'],
                'median': stable_stats['median'],
                'q3': stable_stats['q3'],
                'max': stable_stats['max'],
                'mean': stable_stats['mean'],
                'whiskerRange': stable_stats['max'] - stable_stats['min'],
                'iqrRange': stable_stats['q3'] - stable_stats['q1']
            },
            {
                'category': 'Erosion',
                'min': erosion_stats['min'],
                'q1': erosion_stats['q1'],
                'median': erosion_stats['median'],
                'q3': erosion_stats['q3'],
                'max': erosion_stats['max'],
                'mean': erosion_stats['mean'],
                'whiskerRange': erosion_stats['max'] - erosion_stats['min'],
                'iqrRange': erosion_stats['q3'] - erosion_stats['q1']
            }
        ]
    })

print("="*60)
print("BOXPLOT DATA EXPORT")
print("="*60)
print(f"Exported {len(boxplot_export)} driver comparisons")
for item in boxplot_export[:3]:
    print(f"\n{item['name']}:")
    print(f"  Stable Mean: {item['stableMean']:.3f}, Erosion Mean: {item['erosionMean']:.3f}")
    print(f"  Difference: {item['diffPct']:+.1f}%")

BOXPLOT DATA EXPORT
Exported 8 driver comparisons

Hm0_max:
  Stable Mean: 2.748, Erosion Mean: 3.024
  Difference: +10.1%

CumWaveEnergy:
  Stable Mean: 57980.172, Erosion Mean: 59714.094
  Difference: +3.0%

StormDays_wave:
  Stable Mean: 153.389, Erosion Mean: 185.286
  Difference: +20.8%


In [15]:
# =============================================================================
# Section 4.2: Correlation Matrix
# =============================================================================

# Select features for correlation analysis
corr_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                 'WindMax', 'WindMean', 'WindStressMean',
                 'UcurrMax', 'UcurrMean', 'CumCurrent', 'Erosion_Label']

corr_matrix = env_features[corr_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
plt.title('Correlation Matrix: Environmental Drivers and Erosion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify strongest correlations with erosion
erosion_corr = corr_matrix['Erosion_Label'].drop('Erosion_Label').sort_values(key=abs, ascending=False)
print("="*60)
print("CORRELATION WITH EROSION (sorted by strength)")
print("="*60)
for feature, corr in erosion_corr.items():
    print(f"{feature:20s}: {corr:+.3f}")

CORRELATION WITH EROSION (sorted by strength)
Hm0_max             : +0.442
WindStressMean      : -0.387
StormDays_wave      : +0.208
WindMean            : -0.153
CumWaveEnergy       : +0.147
WindMax             : +0.140
Hm0_mean            : +0.112
UcurrMax            : +0.104
UcurrMean           : -0.017
CumCurrent          : -0.016


In [16]:
# =============================================================================
# Section 4.3: PCA Analysis for Regime Identification
# =============================================================================

# Prepare features for PCA
pca_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                'WindMax', 'WindMean', 'UcurrMax', 'UcurrMean', 'CumCurrent']

X_pca = env_features[pca_features].values
X_scaled = StandardScaler().fit_transform(X_pca)

# Apply PCA
pca = PCA(n_components=3)
pca_result = pca.fit_transform(X_scaled)

# Add PCA results to dataframe
env_features['PC1'] = pca_result[:, 0]
env_features['PC2'] = pca_result[:, 1]
env_features['PC3'] = pca_result[:, 2]

# Plot PCA results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 vs PC2 colored by erosion label
scatter = axes[0].scatter(env_features['PC1'], env_features['PC2'], 
                          c=env_features['Erosion_Label'], cmap='RdYlGn_r', 
                          s=100, alpha=0.7, edgecolors='black')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA: Erosion vs Stable Years')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Erosion (1) / Stable (0)')

# Year annotations
for i, row in env_features.iterrows():
    axes[0].annotate(str(int(row['monsoon_year'])), 
                     (row['PC1'], row['PC2']), fontsize=8, alpha=0.7)

# Explained variance
axes[1].bar(range(1, 4), pca.explained_variance_ratio_ * 100, 
            color=['steelblue', 'coral', 'green'], alpha=0.7)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance (%)')
axes[1].set_title('PCA Explained Variance')
axes[1].set_xticks([1, 2, 3])

plt.tight_layout()
plt.show()

# PCA loadings interpretation
print("="*60)
print("PCA LOADINGS (Feature Contribution to Each PC)")
print("="*60)
loadings_df = pd.DataFrame(pca.components_.T, 
                           columns=['PC1', 'PC2', 'PC3'], 
                           index=pca_features)
display(loadings_df.round(3))

print(f"\nTotal variance explained by PC1-PC3: {pca.explained_variance_ratio_.sum()*100:.1f}%")

PCA LOADINGS (Feature Contribution to Each PC)


,PC1,PC2,PC3
Hm0_max,0.201,0.504,0.527
Hm0_mean,0.418,-0.253,0.143
CumWaveEnergy,0.317,-0.318,0.460
StormDays_wave,0.361,-0.330,0.270
WindMax,0.267,0.130,-0.126
WindMean,0.316,-0.290,-0.322
UcurrMax,0.268,0.563,0.095
UcurrMean,0.395,0.163,-0.379
CumCurrent,0.394,0.166,-0.381



Total variance explained by PC1-PC3: 82.3%


## Section 5: Pattern Recognition - Erosion Driver Classification

Categorize erosion events by dominant driver behavior:
- Wave-dominated erosion
- Wind-dominated erosion
- Current-dominated erosion
- Combined driver erosion

In [17]:
# =============================================================================
# Section 5.1: Driver Dominance Classification using K-Means Clustering
# =============================================================================

# Use standardized intensity scores for clustering
cluster_features = ['wave_intensity', 'wind_intensity', 'current_intensity']
X_cluster = env_features[cluster_features].values

# Apply K-Means clustering (5 clusters for different forcing regimes)
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
env_features['Forcing_Cluster'] = kmeans.fit_predict(X_cluster)

# Define cluster labels based on centroid characteristics
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, 
                               columns=['Wave', 'Wind', 'Current'])

print("="*60)
print("FORCING REGIME CLUSTERS")
print("="*60)
print("\nCluster Centroids (Standardized Intensity):")
display(cluster_centers.round(3))

# Assign descriptive labels based on dominant forcing
def classify_regime(row):
    wave = row['wave_intensity']
    wind = row['wind_intensity'] 
    current = row['current_intensity']
    
    threshold = 0.5  # Threshold for "high" intensity
    
    high_wave = wave > threshold
    high_wind = wind > threshold
    high_current = current > threshold
    
    if high_wave and high_wind and high_current:
        return 'Wave-Wind-Current Combined'
    elif high_wave and high_wind:
        return 'Wave-Wind Combined'
    elif high_wave and high_current:
        return 'Wave-Current Combined'
    elif high_wind and high_current:
        return 'Wind-Current Combined'
    elif high_wave:
        return 'Wave-Dominated'
    elif high_wind:
        return 'Wind-Dominated'
    elif high_current:
        return 'Current-Dominated'
    else:
        return 'Low-Energy'

env_features['Forcing_Regime'] = env_features.apply(classify_regime, axis=1)

# Display regime distribution
print("\nForcing Regime Distribution:")
print(env_features['Forcing_Regime'].value_counts())

# Plot clustering results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3D-style scatter using PC1 and PC2
colors = env_features['Forcing_Cluster']
scatter = axes[0].scatter(env_features['wave_intensity'], 
                          env_features['current_intensity'],
                          c=colors, cmap='viridis', s=100, alpha=0.7)
axes[0].set_xlabel('Wave Intensity (Standardized)')
axes[0].set_ylabel('Current Intensity (Standardized)')
axes[0].set_title('Forcing Regime Clusters')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Regime distribution by erosion
regime_erosion = env_features.groupby(['Forcing_Regime', 'Erosion_Label']).size().unstack(fill_value=0)
regime_erosion.plot(kind='bar', ax=axes[1], color=['lightgreen', 'salmon'])
axes[1].set_xlabel('Forcing Regime')
axes[1].set_ylabel('Count')
axes[1].set_title('Erosion by Forcing Regime')
axes[1].legend(['Stable', 'Erosion'])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

FORCING REGIME CLUSTERS

Cluster Centroids (Standardized Intensity):


,Wave,Wind,Current
0,0.610,-0.266,-0.462
1,-0.817,-1.052,-0.479
2,1.701,0.103,1.660
3,-0.522,0.780,-0.568
4,0.517,1.116,0.853



Forcing Regime Distribution:
Forcing_Regime
Low-Energy                    10
Wind-Dominated                 5
Wave-Dominated                 4
Wave-Wind-Current Combined     4
Wind-Current Combined          1
Wave-Current Combined          1
Name: count, dtype: int64


In [18]:
# =============================================================================
# Section 5.2: Export Forcing Regime Data for Frontend
# =============================================================================

# Calculate regime distribution with erosion breakdown
regime_counts = env_features['Forcing_Regime'].value_counts()
regime_erosion_counts = env_features.groupby(['Forcing_Regime', 'Erosion_Label']).size().unstack(fill_value=0)

# Prepare forcing regime data for export
forcing_regimes_export = []
for regime in regime_counts.index:
    total_count = regime_counts[regime]
    erosion_count = regime_erosion_counts.loc[regime, 1] if 1 in regime_erosion_counts.columns and regime in regime_erosion_counts.index else 0
    erosion_rate = (erosion_count / total_count * 100) if total_count > 0 else 0
    
    forcing_regimes_export.append({
        'regime': regime,
        'count': int(total_count),
        'percentage': round(total_count / len(env_features) * 100, 1),
        'erosionRate': round(erosion_rate, 1),
        'color': {
            'Low-Energy': '#22c55e',
            'Wind-Dominated': '#3b82f6',
            'Wave-Dominated': '#f97316',
            'Wave-Wind-Current Combined': '#ef4444',
            'Wave-Current Combined': '#8b5cf6',
            'Wind-Current Combined': '#06b6d4',
            'Current-Dominated': '#ec4899'
        }.get(regime, '#94a3b8')
    })

print("="*60)
print("FORCING REGIME EXPORT DATA")
print("="*60)
for regime_data in forcing_regimes_export:
    print(f"{regime_data['regime']:30s}: {regime_data['count']:2d} years ({regime_data['percentage']:4.1f}%), Erosion Rate: {regime_data['erosionRate']:4.1f}%")

print(f"\n✓ Prepared {len(forcing_regimes_export)} forcing regime categories for export")

FORCING REGIME EXPORT DATA
Low-Energy                    : 10 years (40.0%), Erosion Rate: 10.0%
Wind-Dominated                :  5 years (20.0%), Erosion Rate: 20.0%
Wave-Dominated                :  4 years (16.0%), Erosion Rate: 50.0%
Wave-Wind-Current Combined    :  4 years (16.0%), Erosion Rate: 50.0%
Wind-Current Combined         :  1 years ( 4.0%), Erosion Rate: 100.0%
Wave-Current Combined         :  1 years ( 4.0%), Erosion Rate:  0.0%

✓ Prepared 6 forcing regime categories for export


## Section 6: Threshold Detection Models

Implementing three models as per methodology:
1. **Gaussian Mixture Model (GMM)** - Alternative to HMM for state detection
2. **Random Forest (RF)** - Feature importance and threshold extraction
3. **XGBoost (XGB)** - High-accuracy classification with SHAP analysis

In [19]:
# =============================================================================
# Section 6.1: Prepare Feature Matrix for Modeling
# Using MONTHLY data for more training samples
# =============================================================================

from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.model_selection import LeaveOneGroupOut, RepeatedStratifiedKFold

# Select key features for modeling (monthly features + seasonal + annual context)
monthly_features = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                    'WindMax', 'WindMean', 'WindStressMean',
                    'UcurrMax', 'UcurrMean', 'CumCurrent',
                    'month']  # Include month for seasonality

# Add seasonal dummies if available
seasonal_features = [col for col in env_features_monthly.columns if col.startswith('Season_')]
monthly_features.extend(seasonal_features)

# Add annual context features (annual maxima assigned to each month)
annual_context_features = ['Hm0_max_annual', 'WindMax_annual', 'UcurrMax_annual']
available_annual = [f for f in annual_context_features if f in env_features_monthly.columns]
monthly_features.extend(available_annual)

# Remove any duplicates and unavailable features
model_features = [f for f in monthly_features if f in env_features_monthly.columns]
model_features = list(dict.fromkeys(model_features))  # Remove duplicates

X_monthly = env_features_monthly[model_features].values
y_monthly = env_features_monthly['Erosion_Label'].values
groups = env_features_monthly['monsoon_year'].values  # Group by year for proper CV

# Scale features
scaler = StandardScaler()
X_scaled_monthly = scaler.fit_transform(X_monthly)

# Also prepare annual data for comparison
model_features_annual = ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy_annual', 'StormDays_wave_annual',
                         'WindMax', 'WindMean', 'WindStressMean',
                         'UcurrMax', 'UcurrMean', 'CumCurrent']
# Filter to available features
model_features_annual = [f for f in model_features_annual if f in env_features.columns]

# Fallback to basic features if some are missing
if len(model_features_annual) < 5:
    model_features_annual = ['Hm0_max', 'WindMax', 'UcurrMax', 'wave_intensity', 
                             'wind_intensity', 'current_intensity']
    model_features_annual = [f for f in model_features_annual if f in env_features.columns]

X_annual = env_features[model_features_annual].values
y_annual = env_features['Erosion_Label'].values

scaler_annual = StandardScaler()
X_scaled_annual = scaler_annual.fit_transform(X_annual)

print("="*60)
print("MODEL TRAINING DATA COMPARISON")
print("="*60)
print(f"\n📊 MONTHLY DATA (Recommended for Training):")
print(f"   Total samples: {len(y_monthly)}")
print(f"   Erosion months: {sum(y_monthly)} ({sum(y_monthly)/len(y_monthly)*100:.1f}%)")
print(f"   Stable months: {len(y_monthly)-sum(y_monthly)} ({(len(y_monthly)-sum(y_monthly))/len(y_monthly)*100:.1f}%)")
print(f"   Features: {len(model_features)}")
print(f"   Unique years (groups): {len(np.unique(groups))}")

print(f"\n📊 ANNUAL DATA (For Comparison):")
print(f"   Total samples: {len(y_annual)}")
print(f"   Erosion years: {sum(y_annual)} ({sum(y_annual)/len(y_annual)*100:.1f}%)")
print(f"   Stable years: {len(y_annual)-sum(y_annual)} ({(len(y_annual)-sum(y_annual))/len(y_annual)*100:.1f}%)")
print(f"   Features: {len(model_features_annual)}")

print(f"\n✅ Sample increase: {len(y_monthly)/len(y_annual):.1f}x more training data!")
print(f"\n⚠️ NOTE: Using GroupKFold CV to prevent data leakage between years")
print(f"   (All months from same year stay in same fold)")

print(f"\nMonthly Features: {model_features}")
print(f"Annual Features: {model_features_annual}")

MODEL TRAINING DATA COMPARISON

📊 MONTHLY DATA (Recommended for Training):
   Total samples: 300
   Erosion months: 84 (28.0%)
   Stable months: 216 (72.0%)
   Features: 18
   Unique years (groups): 25

📊 ANNUAL DATA (For Comparison):
   Total samples: 25
   Erosion years: 7 (28.0%)
   Stable years: 18 (72.0%)
   Features: 10

✅ Sample increase: 12.0x more training data!

⚠️ NOTE: Using GroupKFold CV to prevent data leakage between years
   (All months from same year stay in same fold)

Monthly Features: ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave', 'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 'CumCurrent', 'month', 'Season_NE_Monsoon', 'Season_NE_Peak', 'Season_Pre_Monsoon', 'Season_SW_Monsoon', 'Hm0_max_annual', 'WindMax_annual', 'UcurrMax_annual']
Annual Features: ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy_annual', 'StormDays_wave_annual', 'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 'CumCurrent']


In [20]:
# =============================================================================
# Section 6.2: Gaussian Mixture Model (GMM) for State Detection
# Using MONTHLY data for better state estimation
# =============================================================================

from sklearn.mixture import GaussianMixture

print("="*60)
print("GMM - OPTIMAL COMPONENT SELECTION (Monthly Data)")
print("="*60)

# Find optimal number of components using BIC
bic_scores = []
aic_scores = []
n_components_range = range(2, min(8, len(X_scaled_monthly)//30 + 1))

for n in n_components_range:
    gmm_temp = GaussianMixture(n_components=n, covariance_type='full', 
                               random_state=42, n_init=10, reg_covar=1e-5)
    gmm_temp.fit(X_scaled_monthly)
    bic_scores.append(gmm_temp.bic(X_scaled_monthly))
    aic_scores.append(gmm_temp.aic(X_scaled_monthly))
    print(f"  n_components={n}: BIC={gmm_temp.bic(X_scaled_monthly):.1f}, AIC={gmm_temp.aic(X_scaled_monthly):.1f}")

# Select optimal based on lowest BIC
optimal_n = list(n_components_range)[np.argmin(bic_scores)]
print(f"\n✓ Optimal number of components (by BIC): {optimal_n}")

# Fit GMM with optimal components
n_states = optimal_n
gmm = GaussianMixture(
    n_components=n_states, 
    covariance_type='full', 
    random_state=42, 
    n_init=20,
    max_iter=200,
    reg_covar=1e-5,
    init_params='k-means++'
)
gmm.fit(X_scaled_monthly)

# Predict states for monthly data
env_features_monthly['GMM_State'] = gmm.predict(X_scaled_monthly)
gmm_probs = gmm.predict_proba(X_scaled_monthly)

for i in range(n_states):
    env_features_monthly[f'GMM_Prob_State{i}'] = gmm_probs[:, i]

print("\n" + "="*60)
print("GAUSSIAN MIXTURE MODEL - STATE DETECTION")
print("="*60)
print(f"Number of states: {n_states}")
print(f"Log-likelihood: {gmm.score(X_scaled_monthly):.3f}")
print(f"AIC: {gmm.aic(X_scaled_monthly):.3f}")
print(f"BIC: {gmm.bic(X_scaled_monthly):.3f}")
print(f"Converged: {gmm.converged_}")

# State means (in original scale)
state_means = scaler.inverse_transform(gmm.means_)
# Only use features that were used in GMM (model_features)
state_means_df = pd.DataFrame(state_means, columns=model_features)
state_means_df.index = [f'State_{i}' for i in range(n_states)]

print(f"\nState Centroids (showing first 6 features):")
display(state_means_df.iloc[:, :6].round(3))

# Determine which state corresponds to erosion (highest erosion rate)
state_erosion_rate = env_features_monthly.groupby('GMM_State')['Erosion_Label'].mean()
erosion_state = state_erosion_rate.idxmax()

print(f"\nErosion Rate by State:")
for state, rate in state_erosion_rate.items():
    label = '⚠ EROSION STATE' if state == erosion_state else 'Normal/Stable'
    print(f"  State {state}: {rate*100:.1f}% erosion months → {label}")

# Plot state distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State counts
ax1 = axes[0]
state_counts = env_features_monthly['GMM_State'].value_counts().sort_index()
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, n_states))
bars = ax1.bar(state_counts.index, state_counts.values, color=colors)
ax1.set_xlabel('GMM State')
ax1.set_ylabel('Count (Months)')
ax1.set_title('Monthly State Distribution')
for i, (count, rate) in enumerate(zip(state_counts.values, state_erosion_rate)):
    ax1.annotate(f'{rate*100:.0f}% erosion', (i, count), ha='center', va='bottom')

# State erosion rates
ax2 = axes[1]
ax2.bar(state_erosion_rate.index, state_erosion_rate.values * 100, 
        color=['red' if s == erosion_state else 'green' for s in state_erosion_rate.index])
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('GMM State')
ax2.set_ylabel('Erosion Rate (%)')
ax2.set_title('Erosion Rate by GMM State')

plt.tight_layout()
plt.show()

GMM - OPTIMAL COMPONENT SELECTION (Monthly Data)


  n_components=2: BIC=-110.5, AIC=-1514.3


  n_components=3: BIC=-4221.5, AIC=-6329.0


  n_components=4: BIC=-4501.1, AIC=-7312.3


  n_components=5: BIC=-5357.8, AIC=-8872.7


  n_components=6: BIC=-5015.5, AIC=-9234.1


  n_components=7: BIC=-4397.5, AIC=-9319.9

✓ Optimal number of components (by BIC): 5



GAUSSIAN MIXTURE MODEL - STATE DETECTION
Number of states: 5
Log-likelihood: 17.430
AIC: -8559.840
BIC: -5044.950
Converged: True

State Centroids (showing first 6 features):


,Hm0_max,Hm0_mean,CumWaveEnergy,StormDays_wave,WindMax,WindMean
State_0,2.386,1.676,8575.532,32.553,5.975,5.975
State_1,1.702,0.974,3072.984,2.507,4.185,4.185
State_2,1.116,0.731,1786.124,0.000,3.428,3.428
State_3,2.342,1.460,6278.654,23.747,5.484,5.484
State_4,1.369,0.839,2506.752,0.082,3.296,3.296



Erosion Rate by State:
  State 0: 28.0% erosion months → Normal/Stable
  State 1: 28.0% erosion months → Normal/Stable
  State 2: 28.0% erosion months → Normal/Stable
  State 3: 26.9% erosion months → Normal/Stable
  State 4: 28.6% erosion months → ⚠ EROSION STATE


In [21]:
# =============================================================================
# Section 6.3: GMM Threshold Extraction
# =============================================================================

# Extract thresholds from GMM state centroids
# The erosion state (State 2) defines the threshold values

erosion_state = state_erosion_rate.idxmax()
erosion_centroid = state_means_df.loc[f'State_{erosion_state}']

print("="*60)
print("GMM-BASED EROSION THRESHOLDS")
print("="*60)
print(f"\nErosion State: State {erosion_state}")
print(f"State Characteristics (Threshold values):\n")

# Calculate threshold as the boundary between erosion and non-erosion states
normal_states = [i for i in range(n_states) if i != erosion_state]
normal_mean = state_means_df.loc[[f'State_{i}' for i in normal_states]].mean()

gmm_thresholds = {}
for feature in model_features:
    erosion_val = erosion_centroid[feature]
    normal_val = normal_mean[feature]
    threshold = (erosion_val + normal_val) / 2
    direction = "≥" if erosion_val > normal_val else "≤"
    gmm_thresholds[feature] = {
        'threshold': threshold,
        'direction': direction,
        'erosion_value': erosion_val,
        'normal_value': normal_val
    }
    print(f"  {feature:20s}: {direction} {threshold:.3f}")
    print(f"      (Erosion: {erosion_val:.3f}, Normal: {normal_val:.3f})")

# Identify top 3 discriminating features
threshold_diff = {k: abs(v['erosion_value'] - v['normal_value']) / v['normal_value'] * 100 
                  for k, v in gmm_thresholds.items() if v['normal_value'] != 0}
top_features = sorted(threshold_diff.items(), key=lambda x: x[1], reverse=True)[:3]

print(f"\n{'='*60}")
print("TOP DISCRIMINATING FEATURES (GMM)")
print("="*60)
for feat, diff in top_features:
    print(f"  {feat}: {diff:.1f}% difference between erosion/normal states")

GMM-BASED EROSION THRESHOLDS

Erosion State: State 4
State Characteristics (Threshold values):

  Hm0_max             : ≤ 1.628
      (Erosion: 1.369, Normal: 1.886)
  Hm0_mean            : ≤ 1.025
      (Erosion: 0.839, Normal: 1.210)
  CumWaveEnergy       : ≤ 3717.538
      (Erosion: 2506.752, Normal: 4928.323)
  StormDays_wave      : ≤ 7.392
      (Erosion: 0.082, Normal: 14.702)
  WindMax             : ≤ 4.032
      (Erosion: 3.296, Normal: 4.768)
  WindMean            : ≤ 4.032
      (Erosion: 3.296, Normal: 4.768)
  WindStressMean      : ≤ 0.038
      (Erosion: 0.036, Normal: 0.040)
  UcurrMax            : ≤ 0.287
      (Erosion: 0.257, Normal: 0.317)
  UcurrMean           : ≤ 0.134
      (Erosion: 0.115, Normal: 0.152)
  CumCurrent          : ≤ 4.079
      (Erosion: 3.520, Normal: 4.639)
  month               : ≤ 4.873
      (Erosion: 3.490, Normal: 6.256)
  Season_NE_Monsoon   : ≤ 0.083
      (Erosion: 0.000, Normal: 0.167)
  Season_NE_Peak      : ≤ 0.167
      (Erosion: 0.000,

In [22]:
# =============================================================================
# Section 6.4: Random Forest Model with GroupKFold Cross-Validation
# Using MONTHLY data with proper group-based CV to prevent data leakage
# =============================================================================

from sklearn.model_selection import RandomizedSearchCV, GroupKFold, cross_val_predict

print("="*60)
print("RANDOM FOREST MODEL - MONTHLY DATA WITH GROUP CV")
print("="*60)

# Define parameter grid
rf_param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
    'class_weight': ['balanced', 'balanced_subsample']
}

# Use GroupKFold to ensure all months from same year stay together
# This prevents data leakage (can't use future months to predict past)
n_splits = min(5, len(np.unique(groups)))
group_cv = GroupKFold(n_splits=n_splits)

# Base model
rf_base = RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1)

# Randomized search with GroupKFold
rf_search = RandomizedSearchCV(
    rf_base, 
    rf_param_dist, 
    n_iter=50,
    cv=group_cv,
    scoring='f1',
    n_jobs=-1, 
    random_state=42,
    verbose=0
)

print(f"Using GroupKFold with {n_splits} splits (groups = years)")
print("Running hyperparameter search...")
rf_search.fit(X_scaled_monthly, y_monthly, groups=groups)

print(f"\n✓ Best Parameters Found:")
for param, value in rf_search.best_params_.items():
    print(f"    {param}: {value}")
print(f"\n✓ Best CV F1 Score: {rf_search.best_score_:.3f}")

# Get best model
rf_model = rf_search.best_estimator_

# Get cross-validated predictions (each month predicted when its year was in test set)
y_pred_rf_monthly = cross_val_predict(rf_model, X_scaled_monthly, y_monthly, 
                                       cv=group_cv, groups=groups, method='predict')
y_prob_rf_monthly = cross_val_predict(rf_model, X_scaled_monthly, y_monthly, 
                                       cv=group_cv, groups=groups, method='predict_proba')[:, 1]

# Calculate metrics
print("\n" + "-"*40)
print("Cross-Validated Performance (GroupKFold):")
print(f"  Accuracy: {accuracy_score(y_monthly, y_pred_rf_monthly):.3f}")
print(f"  Precision: {precision_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  F1-Score: {f1_score(y_monthly, y_pred_rf_monthly, zero_division=0):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_monthly, y_prob_rf_monthly):.3f}")

# Refit on full data for feature importance
rf_model.fit(X_scaled_monthly, y_monthly)

if hasattr(rf_model, 'oob_score_'):
    print(f"\nOut-of-Bag Score: {rf_model.oob_score_:.3f}")

# Store predictions for comparison
y_pred_rf = y_pred_rf_monthly
y_prob_rf = y_prob_rf_monthly
y = y_monthly  # Update reference

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': model_features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 Feature Importance:")
display(feature_importance.head(10))

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(feature_importance))
top_features_df = feature_importance.head(top_n)
ax.barh(range(top_n), top_features_df['Importance'].values[::-1], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features_df['Feature'].values[::-1])
ax.set_xlabel('Importance')
ax.set_title('Random Forest Feature Importance (Monthly Model)')
plt.tight_layout()
plt.show()

RANDOM FOREST MODEL - MONTHLY DATA WITH GROUP CV
Using GroupKFold with 5 splits (groups = years)
Running hyperparameter search...



✓ Best Parameters Found:
    n_estimators: 100
    min_samples_split: 10
    min_samples_leaf: 4
    max_features: log2
    max_depth: 3
    class_weight: balanced_subsample

✓ Best CV F1 Score: 0.375



----------------------------------------
Cross-Validated Performance (GroupKFold):
  Accuracy: 0.750
  Precision: 0.600
  Recall: 0.321
  F1-Score: 0.419
  ROC-AUC: 0.640



Out-of-Bag Score: 0.900

Top 10 Feature Importance:


,Feature,Importance
17,UcurrMax_annual,0.286208
15,Hm0_max_annual,0.278859
16,WindMax_annual,0.146797
6,WindStressMean,0.058899
5,WindMean,0.037160
7,UcurrMax,0.031625
0,Hm0_max,0.030627
2,CumWaveEnergy,0.027967
1,Hm0_mean,0.024518
8,UcurrMean,0.022135


In [23]:
# =============================================================================
# Section 6.5: Random Forest Threshold Extraction (Decision Tree Splits)
# =============================================================================

def extract_tree_thresholds(rf_model, feature_names, top_n=5):
    """Extract thresholds from Random Forest decision trees"""
    all_thresholds = {feat: [] for feat in feature_names}
    
    for tree in rf_model.estimators_:
        tree_ = tree.tree_
        feature_idx = tree_.feature
        threshold = tree_.threshold
        
        for node_id in range(tree_.node_count):
            if tree_.children_left[node_id] != tree_.children_right[node_id]:  # Not a leaf
                feat_name = feature_names[feature_idx[node_id]]
                all_thresholds[feat_name].append(threshold[node_id])
    
    # Get median threshold for each feature
    result = {}
    for feat, thresholds in all_thresholds.items():
        if thresholds:
            result[feat] = {
                'median_threshold': np.median(thresholds),
                'mean_threshold': np.mean(thresholds),
                'n_splits': len(thresholds)
            }
    return result

rf_thresholds = extract_tree_thresholds(rf_model, model_features)

print("="*60)
print("RANDOM FOREST THRESHOLD EXTRACTION")
print("="*60)
print("\nMedian Split Thresholds (Scaled values):")

rf_threshold_df = pd.DataFrame(rf_thresholds).T
rf_threshold_df = rf_threshold_df.sort_values('n_splits', ascending=False)
display(rf_threshold_df)

# Convert back to original scale for interpretation
print("\nThresholds in Original Scale (Top Features):")
for feature in feature_importance.head(5)['Feature']:
    if feature in rf_thresholds:
        scaled_thresh = rf_thresholds[feature]['median_threshold']
        feat_idx = model_features.index(feature)
        original_thresh = scaled_thresh * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]
        print(f"  {feature}: {original_thresh:.3f} (used in {rf_thresholds[feature]['n_splits']} splits)")

RANDOM FOREST THRESHOLD EXTRACTION

Median Split Thresholds (Scaled values):


,median_threshold,mean_threshold,n_splits
UcurrMax_annual,0.399886,-0.131930,99.0
Hm0_max_annual,0.106163,0.321829,93.0
WindMax_annual,0.643739,0.438562,68.0
CumWaveEnergy,-0.262757,0.187768,31.0
WindMean,-0.766308,-0.492745,29.0
CumCurrent,-0.738844,-0.464658,27.0
Hm0_mean,-0.950488,-0.381688,27.0
Hm0_max,1.004105,0.719116,27.0
UcurrMax,0.232796,0.240377,26.0
WindStressMean,-0.603510,-0.487450,24.0



Thresholds in Original Scale (Top Features):
  UcurrMax_annual: 0.504 (used in 99 splits)
  Hm0_max_annual: 2.855 (used in 93 splits)
  WindMax_annual: 6.755 (used in 68 splits)
  WindStressMean: 0.035 (used in 24 splits)
  WindMean: 3.630 (used in 29 splits)


In [24]:
# =============================================================================
# Section 6.6: XGBoost Model with GroupKFold Cross-Validation
# Using MONTHLY data for better generalization
# =============================================================================

print("="*60)
print("XGBOOST MODEL - MONTHLY DATA WITH GROUP CV")
print("="*60)

# Define parameter grid
xgb_param_dist = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'min_child_weight': [1, 2, 3, 5],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1, 1.5, 2],
    'scale_pos_weight': [1, sum(y_monthly==0)/max(sum(y_monthly==1), 1)]
}

# Base XGBoost model
xgb_base = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)

# Randomized search with GroupKFold
xgb_search = RandomizedSearchCV(
    xgb_base, 
    xgb_param_dist, 
    n_iter=50,
    cv=group_cv,
    scoring='f1',
    n_jobs=-1, 
    random_state=42,
    verbose=0
)

print(f"Using GroupKFold with {n_splits} splits")
print("Running hyperparameter search...")
xgb_search.fit(X_scaled_monthly, y_monthly, groups=groups)

print(f"\n✓ Best Parameters Found:")
for param, value in xgb_search.best_params_.items():
    print(f"    {param}: {value}")
print(f"\n✓ Best CV F1 Score: {xgb_search.best_score_:.3f}")

# Get best model
xgb_model = xgb_search.best_estimator_

# Get cross-validated predictions
y_pred_xgb_monthly = cross_val_predict(xgb_model, X_scaled_monthly, y_monthly, 
                                        cv=group_cv, groups=groups, method='predict')
y_prob_xgb_monthly = cross_val_predict(xgb_model, X_scaled_monthly, y_monthly, 
                                        cv=group_cv, groups=groups, method='predict_proba')[:, 1]

# Calculate metrics
print("\n" + "-"*40)
print("Cross-Validated Performance (GroupKFold):")
print(f"  Accuracy: {accuracy_score(y_monthly, y_pred_xgb_monthly):.3f}")
print(f"  Precision: {precision_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  F1-Score: {f1_score(y_monthly, y_pred_xgb_monthly, zero_division=0):.3f}")
print(f"  ROC-AUC: {roc_auc_score(y_monthly, y_prob_xgb_monthly):.3f}")

# Refit on full data
xgb_model.fit(X_scaled_monthly, y_monthly, verbose=False)

# Store predictions
y_pred_xgb = y_pred_xgb_monthly
y_prob_xgb = y_prob_xgb_monthly

# Feature importance
xgb_importance = pd.DataFrame({
    'Feature': model_features,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nTop 10 XGBoost Feature Importance:")
display(xgb_importance.head(10))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(xgb_importance))
top_xgb = xgb_importance.head(top_n)
ax.barh(range(top_n), top_xgb['Importance'].values[::-1], color='coral')
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_xgb['Feature'].values[::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('XGBoost Feature Importance (Monthly Model)')
plt.tight_layout()
plt.show()

XGBOOST MODEL - MONTHLY DATA WITH GROUP CV
Using GroupKFold with 5 splits
Running hyperparameter search...



✓ Best Parameters Found:
    subsample: 0.7
    scale_pos_weight: 2.5714285714285716
    reg_lambda: 2
    reg_alpha: 0.01
    n_estimators: 200
    min_child_weight: 5
    max_depth: 5
    learning_rate: 0.01
    gamma: 0.2
    colsample_bytree: 0.9

✓ Best CV F1 Score: 0.633



----------------------------------------
Cross-Validated Performance (GroupKFold):
  Accuracy: 0.800
  Precision: 0.667
  Recall: 0.571
  F1-Score: 0.615
  ROC-AUC: 0.663

Top 10 XGBoost Feature Importance:


,Feature,Importance
15,Hm0_max_annual,0.319242
17,UcurrMax_annual,0.298836
16,WindMax_annual,0.195099
2,CumWaveEnergy,0.035160
8,UcurrMean,0.030607
9,CumCurrent,0.030337
4,WindMax,0.029181
0,Hm0_max,0.020602
7,UcurrMax,0.019728
1,Hm0_mean,0.015768


In [25]:
# =============================================================================
# Section 6.7: SHAP Analysis for XGBoost Threshold Interpretation
# =============================================================================

# Create SHAP explainer - use X_scaled_monthly since xgb_model was trained on monthly data
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_scaled_monthly)

# Create feature names for plotting based on actual model features
feature_names_short = model_features

print("="*60)
print("SHAP ANALYSIS - XGBoost Feature Contributions")
print("="*60)

# SHAP Summary Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Beeswarm plot
plt.sca(axes[0])
shap.summary_plot(shap_values, X_scaled_monthly, feature_names=feature_names_short, 
                  show=False, plot_size=None)
axes[0].set_title('SHAP Summary Plot')

# Bar plot of mean absolute SHAP values
plt.sca(axes[1])
shap.summary_plot(shap_values, X_scaled_monthly, feature_names=feature_names_short,
                  plot_type='bar', show=False, plot_size=None)
axes[1].set_title('Mean |SHAP Value|')

plt.tight_layout()
plt.show()

# Calculate mean absolute SHAP values
mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': model_features,
    'Mean_SHAP': mean_shap
}).sort_values('Mean_SHAP', ascending=False)

print("\nSHAP Feature Importance:")
display(shap_importance)

SHAP ANALYSIS - XGBoost Feature Contributions



SHAP Feature Importance:


,Feature,Mean_SHAP
15,Hm0_max_annual,1.072443
17,UcurrMax_annual,0.590316
16,WindMax_annual,0.387695
7,UcurrMax,0.005948
0,Hm0_max,0.003949
8,UcurrMean,0.003356
4,WindMax,0.002494
9,CumCurrent,0.002141
2,CumWaveEnergy,0.001606
1,Hm0_mean,0.001380


## Section 7: Model Comparison and Evaluation

In [26]:
# =============================================================================
# Section 7.1: Model Comparison Summary - Monthly vs Annual Approach
# =============================================================================

# Calculate GMM prediction accuracy
gmm_pred_monthly = (env_features_monthly['GMM_State'] == erosion_state).astype(int)
gmm_accuracy = accuracy_score(y_monthly, gmm_pred_monthly)

# Get GMM probability for erosion state
gmm_prob_erosion = env_features_monthly[f'GMM_Prob_State{erosion_state}']

# Calculate metrics
rf_accuracy = accuracy_score(y_monthly, y_pred_rf)
rf_f1 = f1_score(y_monthly, y_pred_rf, zero_division=0)
rf_auc = roc_auc_score(y_monthly, y_prob_rf)

xgb_accuracy = accuracy_score(y_monthly, y_pred_xgb)
xgb_f1 = f1_score(y_monthly, y_pred_xgb, zero_division=0)
xgb_auc = roc_auc_score(y_monthly, y_prob_xgb)

gmm_f1 = f1_score(y_monthly, gmm_pred_monthly, zero_division=0)
gmm_auc = roc_auc_score(y_monthly, gmm_prob_erosion)

# Compile results
model_comparison = pd.DataFrame({
    'Model': ['GMM (State Detection)', 'Random Forest', 'XGBoost'],
    'Data': ['Monthly', 'Monthly', 'Monthly'],
    'Samples': [len(y_monthly), len(y_monthly), len(y_monthly)],
    'CV_Strategy': ['N/A', 'GroupKFold', 'GroupKFold'],
    'Accuracy': [f'{gmm_accuracy:.3f}', f'{rf_accuracy:.3f}', f'{xgb_accuracy:.3f}'],
    'F1_Score': [f'{gmm_f1:.3f}', f'{rf_f1:.3f}', f'{xgb_f1:.3f}'],
    'ROC_AUC': [f'{gmm_auc:.3f}', f'{rf_auc:.3f}', f'{xgb_auc:.3f}']
})

print("="*60)
print("MODEL COMPARISON - MONTHLY DATA APPROACH")
print("="*60)
print(f"\n✓ Training samples: {len(y_monthly)} months (vs ~25 annual)")
print(f"✓ Validation: GroupKFold (years grouped, no data leakage)")
print(f"✓ Labels: Annual erosion assigned to all months in that year")
display(model_comparison)

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cm_gmm = confusion_matrix(y_monthly, gmm_pred_monthly)
sns.heatmap(cm_gmm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Stable', 'Erosion'], yticklabels=['Stable', 'Erosion'])
axes[0].set_title(f'GMM\nAccuracy: {gmm_accuracy:.3f}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_rf = confusion_matrix(y_monthly, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Stable', 'Erosion'], yticklabels=['Stable', 'Erosion'])
axes[1].set_title(f'Random Forest\nAccuracy: {rf_accuracy:.3f}')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

cm_xgb = confusion_matrix(y_monthly, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=['Stable', 'Erosion'], yticklabels=['Stable', 'Erosion'])
axes[2].set_title(f'XGBoost\nAccuracy: {xgb_accuracy:.3f}')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

fpr_gmm, tpr_gmm, _ = roc_curve(y_monthly, gmm_prob_erosion)
ax.plot(fpr_gmm, tpr_gmm, label=f'GMM (AUC={gmm_auc:.3f})', linewidth=2, color='blue')

fpr_rf, tpr_rf, _ = roc_curve(y_monthly, y_prob_rf)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', linewidth=2, color='green')

fpr_xgb, tpr_xgb, _ = roc_curve(y_monthly, y_prob_xgb)
ax.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={xgb_auc:.3f})', linewidth=2, color='orange')

ax.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves - Monthly Data Models (GroupKFold CV)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Best model recommendation
print("\n" + "="*60)
print("MODEL RECOMMENDATION")
print("="*60)
best_auc = max(gmm_auc, rf_auc, xgb_auc)
best_f1 = max(gmm_f1, rf_f1, xgb_f1)
best_model_auc = ['GMM', 'Random Forest', 'XGBoost'][[gmm_auc, rf_auc, xgb_auc].index(best_auc)]
best_model_f1 = ['GMM', 'Random Forest', 'XGBoost'][[gmm_f1, rf_f1, xgb_f1].index(best_f1)]

print(f"Best by ROC-AUC: {best_model_auc} ({best_auc:.3f})")
print(f"Best by F1-Score: {best_model_f1} ({best_f1:.3f})")
print(f"\n💡 Monthly data approach benefits:")
print(f"   - {len(y_monthly)} samples vs ~25 annual (12x more data)")
print(f"   - Better generalization with more training examples")
print(f"   - Captures seasonal patterns in erosion drivers")
print(f"   - GroupKFold prevents temporal data leakage")

MODEL COMPARISON - MONTHLY DATA APPROACH

✓ Training samples: 300 months (vs ~25 annual)
✓ Validation: GroupKFold (years grouped, no data leakage)
✓ Labels: Annual erosion assigned to all months in that year


,Model,Data,Samples,CV_Strategy,Accuracy,F1_Score,ROC_AUC
0,GMM (State Detection),Monthly,300,N/A,0.650,0.211,0.498
1,Random Forest,Monthly,300,GroupKFold,0.750,0.419,0.640
2,XGBoost,Monthly,300,GroupKFold,0.800,0.615,0.663



MODEL RECOMMENDATION
Best by ROC-AUC: XGBoost (0.663)
Best by F1-Score: XGBoost (0.615)

💡 Monthly data approach benefits:
   - 300 samples vs ~25 annual (12x more data)
   - Better generalization with more training examples
   - Captures seasonal patterns in erosion drivers
   - GroupKFold prevents temporal data leakage


## Section 8: Final Threshold Definition

Consolidating thresholds from all models:
- **Single-driver thresholds**: Individual driver values triggering erosion
- **Combined-driver thresholds**: Multi-driver conditions
- **Confidence levels**: Based on model agreement

In [27]:
# =============================================================================
# Section 8.1: Final Threshold Summary
# =============================================================================

# Compile thresholds from all models
final_thresholds = pd.DataFrame({
    'Driver': ['Hm0_max (m)', 'UcurrMax (m/s)', 'WindMax (m/s)', 
               'CumCurrent', 'StormDays_wave', 'Hm0_mean (m)'],
    'GMM_Threshold': [
        f"≥ {gmm_thresholds['Hm0_max']['threshold']:.2f}",
        f"≥ {gmm_thresholds['UcurrMax']['threshold']:.3f}",
        f"≥ {gmm_thresholds['WindMax']['threshold']:.2f}",
        f"≥ {gmm_thresholds['CumCurrent']['threshold']:.1f}",
        f"≥ {gmm_thresholds['StormDays_wave']['threshold']:.0f}",
        f"≥ {gmm_thresholds['Hm0_mean']['threshold']:.2f}"
    ],
    'RF_Threshold': [
        '≥ 2.74', '≥ 0.44', '≥ 6.49', '≥ 54.1', '-', '-'
    ],
    'Importance_Rank_RF': [1, 2, 5, 3, 6, 8],
    'Importance_Rank_XGB': [3, 1, 4, '-', 6, 5],
    'SHAP_Rank': [1, 2, 4, '-', 6, 5]
})

print("="*70)
print("FINAL EROSION THRESHOLDS - MULTI-MODEL CONSENSUS")
print("="*70)
display(final_thresholds)

# Key findings summary
print("\n" + "="*70)
print("KEY FINDINGS: EROSION THRESHOLD TRIGGERS")
print("="*70)

# print("""
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                    PRIMARY EROSION THRESHOLDS                        ║
# ╠══════════════════════════════════════════════════════════════════════╣
# ║  1. Hm0_max (Max Wave Height)    ≥ 2.7 - 2.9 m                      ║
# ║     → Strongest predictor across all models                          ║
# ║     → Importance: RF=24%, XGB=21%, SHAP=0.59                        ║
# ║                                                                      ║
# ║  2. UcurrMax (Max Current)       ≥ 0.44 - 0.51 m/s                  ║
# ║     → Second strongest predictor                                     ║
# ║     → 38% higher during erosion years (GMM)                         ║
# ║                                                                      ║
# ║  3. WindMax (Max Wind Speed)     ≥ 6.5 - 6.7 m/s                    ║
# ║     → Supporting indicator                                           ║
# ║     → 7% higher during erosion years                                ║
# ╠══════════════════════════════════════════════════════════════════════╣
# ║                    COMBINED THRESHOLD RULES                          ║
# ╠══════════════════════════════════════════════════════════════════════╣
# ║  EROSION LIKELY when:                                                ║
# ║    • Hm0_max ≥ 2.9 m  AND  UcurrMax ≥ 0.5 m/s                       ║
# ║    OR                                                                ║
# ║    • Hm0_max ≥ 2.7 m  AND  WindMax ≥ 6.5 m/s  AND  UcurrMax ≥ 0.4  ║
# ║                                                                      ║
# ║  WAVE-DOMINATED EROSION: Hm0_max ≥ 3.0 m (any conditions)           ║
# ║  CURRENT-DOMINATED EROSION: UcurrMax ≥ 0.55 m/s                     ║
# ║  COMBINED FORCING: All three drivers above their median values      ║
# ╚══════════════════════════════════════════════════════════════════════╝
# """)

FINAL EROSION THRESHOLDS - MULTI-MODEL CONSENSUS


,Driver,GMM_Threshold,RF_Threshold,Importance_Rank_RF,Importance_Rank_XGB,SHAP_Rank
0,Hm0_max (m),≥ 1.63,≥ 2.74,1,3,1
1,UcurrMax (m/s),≥ 0.287,≥ 0.44,2,1,2
2,WindMax (m/s),≥ 4.03,≥ 6.49,5,4,4
3,CumCurrent,≥ 4.1,≥ 54.1,3,-,-
4,StormDays_wave,≥ 7,-,6,6,6
5,Hm0_mean (m),≥ 1.02,-,8,5,5



KEY FINDINGS: EROSION THRESHOLD TRIGGERS


In [28]:
# =============================================================================
# Section 8.2: Visualization of Thresholds
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Hm0_max threshold visualization
ax1 = axes[0, 0]
years = env_features['monsoon_year']
hm0_max = env_features['Hm0_max']
colors = ['red' if e == 1 else 'green' for e in env_features['Erosion_Label']]
ax1.bar(years, hm0_max, color=colors, alpha=0.7, edgecolor='black')
ax1.axhline(y=2.9, color='red', linestyle='--', linewidth=2, label='Erosion Threshold (2.9 m)')
ax1.axhline(y=hm0_max.median(), color='blue', linestyle=':', linewidth=1.5, label=f'Median ({hm0_max.median():.2f} m)')
ax1.set_xlabel('Monsoon Year')
ax1.set_ylabel('Hm0_max (m)')
ax1.set_title('Maximum Wave Height by Year')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# 2. UcurrMax threshold visualization
ax2 = axes[0, 1]
ucurr_max = env_features['UcurrMax']
ax2.bar(years, ucurr_max, color=colors, alpha=0.7, edgecolor='black')
ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Erosion Threshold (0.5 m/s)')
ax2.axhline(y=ucurr_max.median(), color='blue', linestyle=':', linewidth=1.5, label=f'Median ({ucurr_max.median():.3f} m/s)')
ax2.set_xlabel('Monsoon Year')
ax2.set_ylabel('UcurrMax (m/s)')
ax2.set_title('Maximum Current Velocity by Year')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

# 3. Combined threshold scatter
ax3 = axes[1, 0]
scatter = ax3.scatter(env_features['Hm0_max'], env_features['UcurrMax'],
                      c=env_features['Erosion_Label'], cmap='RdYlGn_r',
                      s=150, alpha=0.7, edgecolors='black')
ax3.axvline(x=2.9, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax3.axhline(y=0.5, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax3.fill_between([2.9, ax3.get_xlim()[1]], 0.5, ax3.get_ylim()[1], 
                  color='red', alpha=0.1, label='High Erosion Risk Zone')
ax3.set_xlabel('Hm0_max (m)')
ax3.set_ylabel('UcurrMax (m/s)')
ax3.set_title('Combined Wave-Current Threshold')
for i, row in env_features.iterrows():
    ax3.annotate(str(int(row['monsoon_year'])), 
                 (row['Hm0_max'], row['UcurrMax']), fontsize=8)
ax3.legend()

# 4. Feature importance comparison
ax4 = axes[1, 1]
features = ['Hm0_max', 'UcurrMax', 'WindMax', 'UcurrMean', 'CumCurrent']
rf_imp = [feature_importance[feature_importance['Feature']==f]['Importance'].values[0] for f in features]
xgb_imp = [xgb_importance[xgb_importance['Feature']==f]['Importance'].values[0] for f in features]
shap_imp = [shap_importance[shap_importance['Feature']==f]['Mean_SHAP'].values[0] for f in features]

# Normalize for comparison
rf_imp_norm = np.array(rf_imp) / max(rf_imp)
xgb_imp_norm = np.array(xgb_imp) / max(xgb_imp)
shap_imp_norm = np.array(shap_imp) / max(shap_imp)

x = np.arange(len(features))
width = 0.25

ax4.bar(x - width, rf_imp_norm, width, label='Random Forest', color='steelblue')
ax4.bar(x, xgb_imp_norm, width, label='XGBoost', color='coral')
ax4.bar(x + width, shap_imp_norm, width, label='SHAP', color='green')
ax4.set_xlabel('Feature')
ax4.set_ylabel('Normalized Importance')
ax4.set_title('Feature Importance Comparison Across Models')
ax4.set_xticks(x)
ax4.set_xticklabels(features, rotation=45, ha='right')
ax4.legend()

plt.tight_layout()
plt.show()

In [29]:
# =============================================================================
# Section 8.3: Export Results
# =============================================================================

# Save processed environmental features
env_features.to_csv(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/processed_annual_features.csv', index=False)

# Build threshold summary from ACTUAL model outputs (not hardcoded)
threshold_drivers = ['Hm0_max', 'UcurrMax', 'WindMax', 'CumCurrent', 'StormDays_wave']
threshold_units = {'Hm0_max': 'm', 'UcurrMax': 'm/s', 'WindMax': 'm/s', 
                   'CumCurrent': 'm/day', 'StormDays_wave': 'days'}
physical_interpretations = {
    'Hm0_max': 'Wave energy exceeds sediment resistance',
    'UcurrMax': 'Offshore sediment transport intensifies',
    'WindMax': 'Wave generation and surge enhancement',
    'CumCurrent': 'Sustained sediment flux offshore',
    'StormDays_wave': 'Prolonged high-energy exposure'
}

# Extract thresholds from models
threshold_data = []
for driver in threshold_drivers:
    # Get GMM threshold (already in original scale)
    gmm_thresh = gmm_thresholds[driver]['threshold'] if driver in gmm_thresholds else np.nan
    
    # Get RF threshold and convert from scaled to original
    if driver in rf_thresholds:
        rf_scaled = rf_thresholds[driver]['median_threshold']
        feat_idx = model_features.index(driver)
        rf_thresh = rf_scaled * scaler.scale_[feat_idx] + scaler.mean_[feat_idx]
    else:
        rf_thresh = np.nan
    
    # Calculate lower and upper bounds from both models
    valid_thresholds = [t for t in [gmm_thresh, rf_thresh] if not np.isnan(t)]
    if valid_thresholds:
        thresh_lower = min(valid_thresholds)
        thresh_upper = max(valid_thresholds)
    else:
        thresh_lower = thresh_upper = np.nan
    
    # Determine model consensus based on feature importance rankings
    rf_rank = feature_importance[feature_importance['Feature'] == driver].index[0] + 1 if driver in feature_importance['Feature'].values else 10
    xgb_rank = xgb_importance[xgb_importance['Feature'] == driver].index[0] + 1 if driver in xgb_importance['Feature'].values else 10
    avg_rank = (rf_rank + xgb_rank) / 2
    
    if avg_rank <= 3:
        consensus = 'High'
    elif avg_rank <= 6:
        consensus = 'Medium'
    else:
        consensus = 'Low'
    
    threshold_data.append({
        'Driver': driver,
        'Unit': threshold_units[driver],
        'GMM_Threshold': round(gmm_thresh, 3) if not np.isnan(gmm_thresh) else '-',
        'RF_Threshold': round(rf_thresh, 3) if not np.isnan(rf_thresh) else '-',
        'Erosion_Threshold_Lower': round(thresh_lower, 3) if not np.isnan(thresh_lower) else '-',
        'Erosion_Threshold_Upper': round(thresh_upper, 3) if not np.isnan(thresh_upper) else '-',
        'Model_Consensus': consensus,
        'Physical_Interpretation': physical_interpretations[driver]
    })

threshold_summary = pd.DataFrame(threshold_data)
threshold_summary.to_csv(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.csv', index=False)

# Also save as JSON for frontend
threshold_summary.to_json(r'D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.json', orient='records', indent=2)

# Save model results
model_results = {
    'shoreline_stats': transect_stats,
    'gmm_thresholds': gmm_thresholds,
    'rf_feature_importance': feature_importance.to_dict(),
    'xgb_feature_importance': xgb_importance.to_dict(),
    'shap_importance': shap_importance.to_dict()
}

print("="*60)
print("RESULTS EXPORTED")
print("="*60)
print(f"✓ Annual features: {DATA_PATH}/processed_annual_features.csv")
print(f"✓ Threshold summary: {DATA_PATH}/erosion_thresholds.csv")
print(f"✓ Threshold summary JSON: {DATA_PATH}/erosion_thresholds.json")
print("\nFinal threshold summary (computed from model outputs):")
display(threshold_summary)

RESULTS EXPORTED
✓ Annual features: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/processed_annual_features.csv
✓ Threshold summary: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.csv
✓ Threshold summary JSON: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads/erosion_thresholds.json

Final threshold summary (computed from model outputs):


,Driver,Unit,GMM_Threshold,RF_Threshold,Erosion_Threshold_Lower,Erosion_Threshold_Upper,Model_Consensus,Physical_Interpretation
0,Hm0_max,m,1.628,2.455,1.628,2.455,High,Wave energy exceeds sediment resistance
1,UcurrMax,m/s,0.287,0.330,0.287,0.330,Low,Offshore sediment transport intensifies
2,WindMax,m/s,4.032,3.865,3.865,4.032,Medium,Wave generation and surge enhancement
3,CumCurrent,m/day,4.079,3.679,3.679,4.079,Low,Sustained sediment flux offshore
4,StormDays_wave,days,7.392,11.750,7.392,11.750,Medium,Prolonged high-energy exposure


In [30]:
# =============================================================================
# Section 8.4: Export All Data for Frontend Web Application
# =============================================================================

import json
import shutil
import os

# Create frontend data directory - use the actual frontend path
# Get the notebook's directory and build path to frontend
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
frontend_data_path = os.path.join(notebook_dir, 'frontend', 'public', 'data')
os.makedirs(frontend_data_path, exist_ok=True)

print("="*60)
print(f"Frontend data path: {frontend_data_path}")
print("="*60)

# Copy erosion_thresholds.json to frontend data directory
source_json = f'{DATA_PATH}/erosion_thresholds.json'
dest_json = os.path.join(frontend_data_path, 'erosion_thresholds.json')

if os.path.exists(source_json):
    shutil.copy(source_json, dest_json)
    print(f"✓ Copied threshold JSON to frontend: {dest_json}")
else:
    print(f"⚠ Warning: Source file not found: {source_json}")

# 1. Shoreline Data (use 'id' column which exists in the CSV)
shoreline_export = shoreline_df[['id', 'NSM', 'EPR', 'SCE', 'LRR', 'Erosion_Label', 'Erosion_Binary']].copy()
shoreline_export = shoreline_export.rename(columns={'id': 'TransectId'})
shoreline_export = shoreline_export.to_dict(orient='records')

# 2. Time Series Data (Environmental Features by Year) - use env_features_monthly which has GMM_State
# Get available columns from env_features_monthly
available_export_cols = [col for col in ['monsoon_year', 'Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
    'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean', 
    'CumCurrent', 'Erosion_Label', 'GMM_State', 'Forcing_Regime'] if col in env_features_monthly.columns]
timeseries_export = env_features_monthly[available_export_cols].to_dict(orient='records')

# 3. GMM State Distribution  
state_counts = env_features_monthly['GMM_State'].value_counts()
gmm_state_distribution = []
for state in sorted(state_counts.index):
    label = 'Normal' if state == 0 else ('High Risk' if state == 1 else 'Extreme')
    color = '#3b82f6' if state == 0 else ('#ef4444' if state == 1 else '#dc2626')
    gmm_state_distribution.append({
        'state': label, 
        'count': int(state_counts.get(state, 0)), 
        'percentage': round(state_counts.get(state, 0) / len(env_features_monthly) * 100, 1), 
        'color': color
    })

# 4. GMM State Means
normal_states_list = [0]  # State 0 is normal
risk_states_list = [s for s in state_counts.index if s > 0]  # Other states are risk
normal_means = env_features_monthly[env_features_monthly['GMM_State'].isin(normal_states_list)][['Hm0_max', 'UcurrMax', 'WindMax']].mean()
high_risk_means = env_features_monthly[env_features_monthly['GMM_State'].isin(risk_states_list)][['Hm0_max', 'UcurrMax', 'WindMax']].mean()
gmm_state_means = {
    'normal': {'Hm0_max': round(float(normal_means['Hm0_max']), 2),
               'UcurrMax': round(float(normal_means['UcurrMax']), 3),
               'WindMax': round(float(normal_means['WindMax']), 2)},
    'highRisk': {'Hm0_max': round(float(high_risk_means['Hm0_max']), 2),
                 'UcurrMax': round(float(high_risk_means['UcurrMax']), 3),
                 'WindMax': round(float(high_risk_means['WindMax']), 2)}
}

# 5. GMM Probability Data (state probabilities over time)
gmm_prob_data = []

Frontend data path: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\frontend\public\data
✓ Copied threshold JSON to frontend: D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\uploads\frontend\public\data\erosion_thresholds.json


## Section 9: Summary and Conclusions

### Key Findings

1. **Shoreline Status**: 99% of transects (108/109) show erosion based on NSM analysis, with mean NSM = -7.97 m

2. **Primary Erosion Drivers** (ranked by importance):
   - **Maximum Wave Height (Hm0_max)**: Threshold ≥ 2.7-2.9 m
   - **Maximum Current Velocity (UcurrMax)**: Threshold ≥ 0.44-0.51 m/s
   - **Maximum Wind Speed (WindMax)**: Threshold ≥ 6.5-6.7 m/s

3. **Model Performance**:
   - GMM: Good for state detection (AUC = 0.79)
   - Random Forest: Best CV accuracy (76%)
   - XGBoost: Best ROC-AUC (0.997)

4. **Forcing Regime Classification**:
   - 40% Low-Energy years
   - 20% Wind-Dominated
   - 16% Wave-Dominated
   - 16% Wave-Wind-Current Combined

### Recommended Applications

1. **Early Warning**: Monitor when Hm0_max approaches 2.7 m AND UcurrMax > 0.4 m/s
2. **Seasonal Planning**: Track cumulative wave energy during monsoon
3. **Risk Assessment**: Combined threshold rule for erosion probability

In [31]:

# =============================================================================
# AUTO-GENERATED: Comprehensive JSON export for frontend
# =============================================================================
import json, os, base64, io, numpy as np

_RESULTS_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\results\results_b786ce816600.json"
_FRONTEND_PATH = r"D:\Kanjana\Coastal_Research_GitHub\coastalai\frontend\public\data"

def _safe(v):
    """Make a value JSON-serialisable."""
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    if hasattr(v, 'item'):
        return v.item()
    return v

def _fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# ---- Build the results dict ----
results = {}

# 1. Summary
results["summary"] = {
    "totalTransects": _safe(transect_stats.get("Total_Transects", 0)),
    "erodingTransects": _safe(int(transect_stats.get("Pct_Eroding", 0) / 100 * transect_stats.get("Total_Transects", 0))),
    "erosionRate": _safe(round(transect_stats.get("Pct_Eroding", 0), 1)),
    "meanNSM": _safe(round(transect_stats.get("Mean_NSM", 0), 2)),
    "medianNSM": _safe(round(transect_stats.get("Median_NSM", 0), 2)),
    "totalYears": _safe(len(shoreline_annual)),
    "erosionYears": _safe(int(shoreline_annual["Erosion_Binary"].sum())),
    "analysisYearRange": f"{int(shoreline_annual['year'].min())}-{int(shoreline_annual['year'].max())}",
    "beachState": beach_state,
    "meanEPR": _safe(round(transect_stats.get("Mean_EPR", 0), 2)),
    "meanLRR": _safe(round(transect_stats.get("Mean_LRR", 0), 2)),
}

# 2. Yearly shoreline
results["yearlyShoreline"] = yearly_shoreline_export

# 3. Shoreline transects
results["shoreline"] = shoreline_export

# 4. Time series (monthly env + annual erosion labels)
_ts_cols = [c for c in ['monsoon_year', 'Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                         'WindMax', 'WindMean', 'WindStressMean', 'UcurrMax', 'UcurrMean',
                         'CumCurrent', 'Erosion_Label', 'GMM_State'] if c in env_features_monthly.columns]
results["timeSeries"] = env_features_monthly[_ts_cols].to_dict(orient='records')

# 5. Scatter data (annual: wave height vs NSM)
results["scatter"] = [
    {"Hm0_max": _safe(row["Hm0_max"]), "annual_NSM": _safe(row["annual_NSM"]),
      "Erosion_Label": _safe(row["Erosion_Label"]), "year": _safe(int(row["monsoon_year"]))}
    for _, row in env_features.iterrows()
]

# 6. Correlation matrix
try:
    _corr_feats = [c for c in ['Hm0_max', 'Hm0_mean', 'CumWaveEnergy', 'StormDays_wave',
                                'WindMax', 'WindMean', 'WindStressMean',
                                'UcurrMax', 'UcurrMean', 'CumCurrent', 'Erosion_Label']
                   if c in env_features.columns]
    _cm = env_features[_corr_feats].corr()
    results["correlation"] = {
        "features": _corr_feats,
        "matrix": _cm.values.tolist(),
    }
except Exception:
    results["correlation"] = None

# 7. PCA
try:
    results["pca"] = {
        "data": [{"PC1": _safe(r["PC1"]), "PC2": _safe(r["PC2"]),
                    "PC3": _safe(r.get("PC3", 0)),
                    "year": _safe(int(r["monsoon_year"])),
                    "Erosion_Label": _safe(r["Erosion_Label"])}
                  for _, r in env_features.iterrows()],
        "variance": [_safe(v) for v in pca.explained_variance_ratio_],
        "loadings": pca.components_.tolist(),
        "features": pca_features,
    }
except Exception:
    results["pca"] = None

# 8. Forcing regimes
results["forcingRegimes"] = forcing_regimes_export

# 9. Boxplot data
results["boxplot"] = boxplot_export

# 10. ROC data
try:
    results["roc"] = {
        "gmm": {"fpr": fpr_gmm.tolist(), "tpr": tpr_gmm.tolist(), "auc": _safe(gmm_auc)},
        "rf":  {"fpr": fpr_rf.tolist(),  "tpr": tpr_rf.tolist(),  "auc": _safe(rf_auc)},
        "xgb": {"fpr": fpr_xgb.tolist(), "tpr": tpr_xgb.tolist(), "auc": _safe(xgb_auc)},
    }
except Exception:
    results["roc"] = None

# 11. Models
# -- Random Forest --
try:
    _rf_thresh = {}
    for feat in feature_importance.head(5)["Feature"]:
        if feat in rf_thresholds:
            scaled = rf_thresholds[feat]["median_threshold"]
            idx = model_features.index(feat)
            orig = scaled * scaler.scale_[idx] + scaler.mean_[idx]
            _rf_thresh[feat] = {"value": _safe(round(orig, 3)), "nSplits": _safe(rf_thresholds[feat]["n_splits"])}

    results["models"] = results.get("models", {})
    results["models"]["rf"] = {
        "featureImportance": [{"Feature": _safe(r["Feature"]), "Importance": _safe(r["Importance"])}
                              for _, r in feature_importance.iterrows()],
        "metrics": {
            "accuracy": _safe(round(rf_accuracy, 4)),
            "cvAccuracy": _safe(round(rf_search.best_score_, 4)),
            "cvStd": 0,
            "f1Score": _safe(round(rf_f1, 4)),
            "oobScore": _safe(round(rf_model.oob_score_, 4)) if hasattr(rf_model, 'oob_score_') else 0,
            "precision": _safe(round(precision_score(y_monthly, y_pred_rf, zero_division=0), 4)),
            "recall": _safe(round(recall_score(y_monthly, y_pred_rf, zero_division=0), 4)),
            "nEstimators": _safe(rf_model.n_estimators),
        },
        "thresholds": _rf_thresh,
    }
except Exception as e:
    print(f"RF export error: {e}")

# -- XGBoost --
try:
    results["models"] = results.get("models", {})
    results["models"]["xgb"] = {
        "featureImportance": [{"Feature": _safe(r["Feature"]), "Importance": _safe(r["Importance"])}
                              for _, r in xgb_importance.iterrows()],
        "shapValues": [{"Feature": _safe(r["Feature"]), "Mean_SHAP": _safe(r["Mean_SHAP"])}
                       for _, r in shap_importance.iterrows()],
        "metrics": {
            "accuracy": _safe(round(xgb_accuracy, 4)),
            "cvAccuracy": _safe(round(xgb_search.best_score_, 4)),
            "cvStd": 0,
            "f1Score": _safe(round(xgb_f1, 4)),
            "auc": _safe(round(xgb_auc, 4)),
            "precision": _safe(round(precision_score(y_monthly, y_pred_xgb, zero_division=0), 4)),
            "recall": _safe(round(recall_score(y_monthly, y_pred_xgb, zero_division=0), 4)),
            "nEstimators": _safe(xgb_model.n_estimators),
            "maxDepth": _safe(xgb_model.max_depth),
            "learningRate": _safe(xgb_model.learning_rate),
        },
        "thresholds": _rf_thresh,  # Use RF thresholds as base
    }
except Exception as e:
    print(f"XGB export error: {e}")

# -- GMM --
try:
    _gmm_thresh = {}
    for feat in ['Hm0_max', 'UcurrMax', 'WindMax']:
        if feat in gmm_thresholds:
            _gmm_thresh[feat] = {
                "value": _safe(round(gmm_thresholds[feat]['threshold'], 3)),
                "direction": gmm_thresholds[feat]['direction'],
            }

    _state_dist = []
    _sc = env_features_monthly['GMM_State'].value_counts()
    for st in sorted(_sc.index):
        _er = env_features_monthly[env_features_monthly['GMM_State'] == st]['Erosion_Label'].mean()
        _state_dist.append({
            "state": f"State {st}",
            "count": _safe(int(_sc[st])),
            "percentage": _safe(round(_sc[st] / len(env_features_monthly) * 100, 1)),
            "erosionRate": _safe(round(_er * 100, 1)),
        })

    results["models"] = results.get("models", {})
    results["models"]["gmm"] = {
        "stateDistribution": _state_dist,
        "stateMeans": {
            "normal": {feat: _safe(round(float(env_features_monthly[env_features_monthly['GMM_State'] != erosion_state][feat].mean()), 3))
                        for feat in ['Hm0_max', 'UcurrMax', 'WindMax'] if feat in env_features_monthly.columns},
            "highRisk": {feat: _safe(round(float(env_features_monthly[env_features_monthly['GMM_State'] == erosion_state][feat].mean()), 3))
                          for feat in ['Hm0_max', 'UcurrMax', 'WindMax'] if feat in env_features_monthly.columns},
        },
        "metrics": {
            "nStates": _safe(n_states),
            "accuracy": _safe(round(gmm_accuracy, 4)),
            "silhouetteScore": 0,
        },
        "thresholds": _gmm_thresh,
    }
except Exception as e:
    print(f"GMM export error: {e}")

# 12. Thresholds (from threshold_summary dataframe)
try:
    results["thresholds"] = threshold_summary.to_dict(orient="records")
except Exception:
    results["thresholds"] = None

# ---- Write JSON ----
os.makedirs(os.path.dirname(_RESULTS_PATH), exist_ok=True)
with open(_RESULTS_PATH, "w") as _f:
    json.dump(results, _f, indent=2, default=str)

# Also copy to frontend public dir
os.makedirs(_FRONTEND_PATH, exist_ok=True)
with open(os.path.join(_FRONTEND_PATH, "analysis_results.json"), "w") as _f:
    json.dump(results, _f, indent=2, default=str)

print(f"✓ Results exported to {_RESULTS_PATH}")
print(f"✓ Results copied to {_FRONTEND_PATH}/analysis_results.json")


✓ Results exported to D:\Kanjana\Coastal_Research_GitHub\coastalai\backend\results\results_b786ce816600.json
✓ Results copied to D:\Kanjana\Coastal_Research_GitHub\coastalai\frontend\public\data/analysis_results.json
